# Document Loading

In [1]:
import hashlib # Used for creating chunk id
import numpy as np # Used for images .
import pymupdf # For PDF loading
from pathlib import Path # For Loading the directory or PDF path .
from typing import List,Dict
import traceback # Used for error handling .

In [2]:
# The following function is used to check if the image has detailing since if there is no much variance
# (basically flat image) it removes it.
# threshold -> minimum variance value.
def is_low_variance(pix, threshold: int = 5) -> bool:
    try:
        if pix is None or pix.samples is None:  # Checking if image is empty.
            return True

        samples = np.frombuffer(pix.samples, dtype=np.uint8)  # Storing pixels (raw bytes) into numpy array.

        if len(samples) == 0:  # Checking if image array is empty.
            return True

        if pix.n >= 3:
            samples = samples.reshape(-1, pix.n)[:, :3].mean(axis=1)  # Converting to grayscale

        return samples.std() < threshold  # Returns True if standard deviation is less than threshold.
    except Exception as e:
        print(f"Error in is_low_variance: {e}")
        return True

In [3]:
# The following function checks if image is mostly white.
# threshold -> 245 -> near white
# ratio -> The threshold ratio of white pixels in the whole picture.
def is_mostly_white(pix, threshold=245, ratio=0.98) -> bool:
    try:
        if pix is None or pix.samples is None:  # Checking if image is empty.
            return True

        samples = np.frombuffer(pix.samples, dtype=np.uint8)  # Storing pixels (raw bytes) into numpy array.

        if len(samples) == 0:  # Checking if image array is empty.
            return True

        if pix.n >= 3:
            samples = samples.reshape(-1, pix.n)[:, :3].mean(axis=1)
        white_pixels = np.sum(samples > threshold)  # Number of near-white pixels in image.

        return (white_pixels / len(samples)) > ratio  # Returns True if most pixels are near white.
    except Exception as e:
        print(f"Error in is_mostly_white: {e}")
        return True

In [4]:
# The following function checks if image has extreme aspect ratio (too wide or too tall) which are usually headers/footers.
# min_ratio -> minimum acceptable aspect ratio
# max_ratio -> maximum acceptable aspect ratio
def is_extreme_aspect_ratio(pix, min_ratio: float = 0.1, max_ratio: float = 10.0) -> bool:
    try:
        if pix is None:  # Checking if image is empty.
            return True

        aspect_ratio = pix.width / max(pix.height, 1)  # Calculating aspect ratio.

        return aspect_ratio < min_ratio or aspect_ratio > max_ratio  # Returns True if aspect ratio is extreme.
    except Exception as e:
        print(f"Error in is_extreme_aspect_ratio: {e}")
        return True

In [5]:
def loading_pdf(dir_path: str = '../data/pdf') -> List[Dict]:  # Return type.
    dir_path = Path(dir_path)  # Loading directory Path
    if not dir_path.is_dir():  # Checking if the directory is valid.
        raise NotADirectoryError(f"{dir_path} is an invalid directory.")

    print(f"\n{'='*80}")
    print(f"  PDF LOADING")
    print(f"{'='*80}")
    print(f"  Directory: {dir_path}")

    pdf_files = list(dir_path.rglob("*.pdf"))  # Storing all the PDFs' paths into list.
    print(f"  Found {len(pdf_files)} PDF file(s)")

    if len(pdf_files) == 0:  # Checking if any PDFs exist in the directory.
        print('  WARNING: No documents found in directory')
        return []

    # All these variables used for stats check at the end.
    all_pdf_size = 0.0
    all_pages = []
    failed_pdf = []

    print(f"\n  Loading PDFs...")
    print("  " + "-" * 76)

    for serial, pdf_path in enumerate(pdf_files, start=1):  # Iterating through all PDFs in directory.
        print(f"  [{serial}/{len(pdf_files)}] Loading: {pdf_path.name}")
        pdf_size_bytes = pdf_path.stat().st_size
        pdf_size_mb = pdf_size_bytes / (1024 ** 2)  # Calculating size of the PDF.
        print(f"       Size: {pdf_size_mb:.2f} MB")

        pdf = None  # For cleanup
        try:
            image_dir = Path('../data/images_pymupdf') / pdf_path.stem  # Directory for storing images in the PDF.
            image_dir.mkdir(parents=True, exist_ok=True)
            pdf = pymupdf.open(filename=pdf_path, filetype="pdf")  # Loading PDF.

            for page_num, page in enumerate(pdf, start=1):
                text_blocks = []  # Used for storing details about blocks of a page.
                page_images = []  # Used for storing images of current page.
                seen_xrefs = set()
                images = page.get_images(full=True)  # Extracting images.

                for img_index, img in enumerate(images):  # Extracting images.
                    pix = None  # For cleanup
                    try:
                        if img[1] != 0:  # Skip soft mask.
                            # soft mask -> Transparency layer
                            continue
                        xref = img[0]
                        if xref in seen_xrefs:  # Checking if the same images are being stored
                            continue
                        seen_xrefs.add(xref)
                        rects = page.get_image_rects(xref)  # Used for getting image edges.
                        if not rects:  # Checking if coordinates or image is empty.
                            continue

                        pix = pymupdf.Pixmap(pdf, xref)  # xref is used to find position of image in PDF.
                        if pix.width < 50 or pix.height < 50:  # Removing very tiny images.
                            pix = None
                            continue
                        if pix.alpha and pix.samples is not None:  # Removing fully transparent images.
                            if max(pix.samples) == 0 and len(pix.samples) > 0:
                                continue
                        if pix.n > 4:
                            pix = pymupdf.Pixmap(pymupdf.csRGB, pix)
                        if is_mostly_white(pix):  # Checking if the image is mostly white.
                            pix = None
                            continue
                        if is_low_variance(pix):  # Checking if it's a flat image.
                            pix = None
                            continue
                        if is_extreme_aspect_ratio(pix):  # Checking if aspect ratio is extreme (headers/footers).
                            pix = None
                            continue

                        img_path = image_dir / f"page_{page_num}_img_{img_index}.png"  # Location for storing images in local disk.
                        pix.save(img_path)  # Saving images in local disk.
                        pix = None
                        rect = rects[0]  # Changed the method since we needed only approx coordinates and not all approx coordinates to be merged, using a FOR loop made multiple copy of image.
                        page_images.append({
                            "image_id": f"{pdf_path.stem}_p{page_num}_i{img_index}",
                            "path": str(img_path),
                            "page": page_num,
                            "bbox": [rect.x0, rect.y0, rect.x1, rect.y1]
                        })  # For metadata.
                    except Exception as img_error:
                        print(f"       Error processing image {img_index}: {img_error}")
                    finally:
                        if pix is not None:
                            pix = None

                # Following loop is to extract texts from a page.
                blocks = sorted(page.get_text("blocks"), key=lambda b: (b[1], b[0]))
                for block_id, b in enumerate(blocks):
                    x0, y0, x1, y1, text = b[:5]  # Coordinates and text of text block.
                    text = text.strip()
                    if len(text) < 20:  # If texts are smaller it is removed since smaller texts can not be very useful.
                        continue
                    block_bbox = [x0, y0, x1, y1]  # Coordinates of the text blocks. Used while checking relevance of image and text.
                    text_blocks.append({
                        "block_id": block_id,
                        "text": text,
                        "bbox": block_bbox,
                        "page": page_num,
                    })  # Used while appending metadata.

                all_pages.append({
                    "source": pdf_path.name,
                    "page": page_num,
                    "text_blocks": text_blocks,
                    "images": page_images
                })

            all_pdf_size += pdf_size_mb
            pdf.close()
            print(f"       ✓ Extracted {len(text_blocks)} text blocks, {len(page_images)} images")

        except Exception as e:
            print(f"       ✗ Error loading {pdf_path.name}: {e}")  # Exception handling.
            failed_pdf.append(pdf_path.name)  # Storing the PDF failed to load.
            traceback.print_exc()  # Used to trace failures similar to python interpreter stack trace.
        finally:
            if pdf is not None and not pdf.is_closed:
                pdf.close()

    # Some stats of Loading all the PDF in a directory.
    print("  " + "-" * 76)
    print(f"\n  SUMMARY:")
    print(f"       Total size: {all_pdf_size:.2f} MB")
    print(f"       Total pages extracted: {len(all_pages)}")
    print(f"       Successful: {len(pdf_files) - len(failed_pdf)}/{len(pdf_files)}")

    # Printing all the PDF which were not able to load.
    if failed_pdf:
        print(f"\n  Failed PDFs:")
        for fp in failed_pdf:
            print(f"       - {fp}")

    print(f"\n{'='*80}\n")
    return all_pages  # Returning the loaded pages.

# Chunking

In [6]:
from langchain_core.documents import Document # Datatype of a block or a chunk .
from typing import List # Used to store list of Documents or to specify return type .
from typing import Tuple
import tiktoken

In [7]:
# The following function is to calculate distance between two blocks and a threshold is set such that
# if two blocks are far those both blocks are separated with different chunks.
def vertical_gap(block1, block2) -> float:
    return block2["bbox"][1] - block1["bbox"][3]  # Distance between bottom of block 1 and top of block 2.

In [8]:
# The following function is used for getting outermost edge of all the chunks combined.
def merge_bbox(blocks):
    if not blocks:
        return None

    return (
        min(b["bbox"][0] for b in blocks),  # x0 left
        min(b["bbox"][1] for b in blocks),  # y0 top
        max(b["bbox"][2] for b in blocks),  # x1 right
        max(b["bbox"][3] for b in blocks)   # y1 bottom
    )

In [9]:
# The following function creates a constant chunk id for same text.
def stable_chunk_id(source: str, page_num: int, text: str) -> str:
    h = hashlib.md5(text.encode("utf-8")).hexdigest()[:8]
    return f"{source}_p{page_num}_c{h}"

In [10]:
# The following function is used to check if a block is relevant to another block using coordinates.
def bbox_overlap(a, b) -> bool:
    return not (
        a[2] < b[0] or  # right edge of a and left edge of b
        a[0] > b[2] or  # left edge of a and right edge of b
        a[3] < b[1] or  # bottom edge of a and top edge of b
        a[1] > b[3]     # top edge of a and bottom edge of b
    )

In [11]:
# The following function helps to identify if the text block near the image is caption of the image based on the coordinates and length of the text.
def is_caption_block(text_block: Dict, image: Dict, max_words: int = 60, max_vertical_dist: int = 80) -> bool:
    text = text_block.get("text", "")  # Text

    if not text or len(text.split()) > max_words:  # Checking if the text is large.
        return False

    tb_bbox = text_block.get("bbox")  # Fetching text block bbox.
    im_bbox = image.get("bbox")  # Fetching image block bbox.

    if not tb_bbox or not im_bbox:  # Checking if bbox is empty.
        return False

    tb_x0, tb_y0, tb_x1, tb_y1 = tb_bbox  # Coordinates of text block.
    im_x0, im_y0, im_x1, im_y1 = im_bbox  # Coordinates of image.

    horizontal_overlap = not (tb_x1 < im_x0 or tb_x0 > im_x1)  # Check for horizontal overlap.

    vertical_distance = min(  # Vertical distance between text block and image.
        abs(tb_y0 - im_y1),
        abs(im_y0 - tb_y1)
    )
    return horizontal_overlap and vertical_distance <= max_vertical_dist  # Returns False if any condition fails.

In [12]:
# The following function gets images that overlap with a chunk's bbox.
def get_overlapping_images(chunk_bbox: Tuple, page_images: List[Dict], vertical_tolerance: int = 200, horizontal_tolerance: int = 50) -> List[str]:
    if not chunk_bbox or not page_images:
        return []

    overlapping_ids = []

    chunk_x0, chunk_y0, chunk_x1, chunk_y1 = chunk_bbox

    for img in page_images:
        img_bbox = img.get("bbox")
        if not img_bbox:
            continue

        img_x0, img_y0, img_x1, img_y1 = img_bbox

        # Check for horizontal overlap or proximity
        horizontal_overlap = not (
            chunk_x1 + horizontal_tolerance < img_x0 or
            chunk_x0 > img_x1 + horizontal_tolerance
        )

        # Calculate vertical distance between chunk and image
        vertical_distance = min(
            abs(chunk_y0 - img_y1),  # Distance from chunk top to image bottom
            abs(img_y0 - chunk_y1),  # Distance from image top to chunk bottom
            abs(chunk_y0 - img_y0),  # Distance between tops
            abs(chunk_y1 - img_y1)   # Distance between bottoms
        )

        # Method 1: Check if horizontally aligned and vertically close
        if horizontal_overlap and vertical_distance <= vertical_tolerance:
            overlapping_ids.append(img["image_id"])
        # Method 2: Or if they actually overlap perfectly
        elif bbox_overlap(chunk_bbox, img_bbox):
            overlapping_ids.append(img["image_id"])

    return overlapping_ids

In [13]:
# Build image objects, mainly used for embedding using openclip and also can be used parallely with text block in graph db.
def build_image_objects(pages: List[Dict]) -> List[Dict]:
    image_objects = []

    # Getting text block and images from the page. (For reference this are the pages already loaded from pdf_loading.)
    for page in pages:
        source = page.get("source", "unknown")
        page_num = page.get("page", 0)
        text_blocks = page.get("text_blocks", [])
        images = page.get("images", [])

        # For every text block in page checking if the image is relevant or caption to it.
        for img in images:
            caption_blocks = [
                tb["text"] for tb in text_blocks
                if tb.get("text") and is_caption_block(tb, img)
            ]

            caption_text = " ".join(caption_blocks).strip() or None  # Caption gathered from text blocks.

            # Appending image objects.
            image_objects.append({
                "image_id": img.get("image_id", "unknown"),
                "type": "image",
                "modality": "vision",
                "source": source,
                "page_num": page_num,
                "bbox": img.get("bbox"),
                "path": img.get("path"),
                "caption_text": caption_text
            })

    return image_objects  # List of image objects with relevant metadata.

In [14]:
# The following function is used create a chunk by adding
def build_chunk(source: str, page_num: int, blocks: List, page_images: List[Dict], related_image_ids: List[str] | None = None) -> Document:
    if not blocks:
        return None

    chunk_text = "\n".join(b.get("text", "") for b in blocks)  # Combing all the texts from the blocks.

    if not chunk_text.strip():  # Checking if the text blocks are empty.
        return None

    chunk_bbox = merge_bbox(blocks)  # Used to get overall chunk coordinates.

    if related_image_ids is None:  # If no related images then fetch related images based on bbox overlap.
        related_image_ids = get_overlapping_images(chunk_bbox, page_images)

    return Document(
        page_content=chunk_text,
        metadata={
            "source": source,
            "page_num": page_num,
            "bbox": chunk_bbox,
            "chunk_id": stable_chunk_id(source, page_num, chunk_text),
            "related_image_ids": related_image_ids or []
        }
    )  # Adding metadata.

In [15]:
# The following function is main chunking strategy, it uses bbox , max characters to chunk different blocks together.
# max_tokens -> maximum tokens in a single chunk (improved from char-based).
# (Replaced char-based with token-based using tiktoken for better semantics.)

# max_vertical_gap -> maximum vertical height between two blocks.
# Calculated using bbox.

# overlap_tokens -> number of tokens to overlap between consecutive chunks
def bbox_chunker(
    pages: List[Dict],
    max_tokens: int = 384,
    max_vertical_gap: int = 60,
    overlap_tokens: int = 96,
    token_model: str = "cl100k_base"
) -> List[Document]:
    tokenizer = tiktoken.get_encoding(token_model)  # For token counting

    all_chunks = []  # Used to store chunks.

    # Extracting data from a dictionary.
    for page in pages:
        source = page.get("source", "")
        page_num = page.get("page", 0)
        blocks = page.get("text_blocks", [])
        page_images = page.get("images", [])  # Getting images in a page.

        if not blocks:
            continue

        i = 0
        while i < len(blocks):
            current_blocks = []  # Used to store blocks to store in a chunk.
            current_tokens = 0  # Calculating maximum tokens in chunk.
            start_i = i  # Used to prevent infinite loop during overlap

            # Building a chunk until max_tokens or vertical gap threshold
            while i < len(blocks):
                block = blocks[i]
                text = block.get("text", "")

                if not text:
                    i += 1
                    continue

                block_tokens = len(tokenizer.encode(text))  # Token count for the block.

                if current_blocks:
                    gap = vertical_gap(current_blocks[-1], block)
                else:
                    gap = 0  # Basically first block of a page.

                # Checking conditions for creating a chunk.
                # (Basically threshold that chunk should have certain number of tokens and text block distances.)
                if current_blocks and (
                    current_tokens + block_tokens > max_tokens
                    or gap > max_vertical_gap
                ):
                    break

                current_blocks.append(block)
                current_tokens += block_tokens
                i += 1

            # Creating chunk from collected blocks.
            if current_blocks:
                chunk = build_chunk(source, page_num, current_blocks, page_images)
                if chunk:
                    all_chunks.append(chunk)  # Creating a chunk and appending it.

            # Step back to create overlap for next chunk.
            if overlap_tokens > 0 and i < len(blocks):
                overlap_tok = 0
                step_back = 0

                # Counting how many blocks to include in overlap.
                for j in range(len(current_blocks) - 1, -1, -1):
                    block_text = current_blocks[j].get("text", "")
                    block_tok = len(tokenizer.encode(block_text))
                    if overlap_tok + block_tok <= overlap_tokens:
                        overlap_tok += block_tok
                        step_back += 1
                    else:
                        break

                # Move index back safely (avoid infinite loop).
                i = max(start_i + 1, i - step_back)

    print(f"  Created {len(all_chunks)} chunks from {len(pages)} pages (max_tokens={max_tokens}, overlap_tokens={overlap_tokens})")

    return all_chunks

# Embedding

In [16]:
import numpy as np # Used for storing embeddings .
import torch # For device selection , model execution and tensor operation .
from PIL import Image # Used for image creation and other image operation .
import open_clip # Image embedding model .
from typing import List, Dict,Tuple # Used for type return .
from dotenv import load_dotenv # Used for loading Huggingface api .
from time import perf_counter
import os

C:\RAKSHAK\MY CODE\AI\rag-pdf\rag-env\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [17]:
load_dotenv() # For loading Huggingface api .

True

In [18]:
# Removing unwanted warnings from sentence transformers library .(Optional)
import warnings
from transformers import logging

logging.set_verbosity_error()
warnings.filterwarnings("ignore", message=".*position_ids.*UNEXPECTED.*")

In [19]:
# Used for loading embedding model and embedding images with their caption .
class ImageEmbeddingModel:
    def __init__(self, model_name: str = "ViT-L-14", pretrained: str = "laion2b_s32b_b82k"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"  # Used to check if the machine has GPU; if not, assign CPU.
        try:
            self.model, self.preprocess, _ = open_clip.create_model_and_transforms(
                model_name=model_name,
                pretrained=pretrained
            )  # Getting essential functions from model.
            self.tokenizer = open_clip.get_tokenizer(model_name)
        except Exception as e:
            raise RuntimeError(f"Failed to load OpenCLIP model: {e}")
        self.model = self.model.to(self.device)  # Device selection for model operations.
        self.model.eval()  # Loading model in eval mode.
        # Displaying device on which model will run.
        if self.device == "cuda":
            print(f"OpenCLIP running on {torch.cuda.get_device_name(0)}.")
        else:
            print("OpenCLIP running on CPU.")
        print(f"Embedding dimension of {model_name} is {open_clip.get_model_config(model_name)['embed_dim']}")

    # Using no_grad since by default torch assumes it is for training and does backprop; here only forward pass is needed.
    @torch.no_grad()
    def embed_image(self, image_objects: List[Dict]) -> Tuple:
        if not image_objects:
            raise ValueError("No images in the image objects.")

        image_objects_embeddings = []  # Used for storing image embeddings.
        start_time = perf_counter()  # Start the timer before image embedding starts .
        # M1 (Embedding time )

        for image_object in image_objects:  # Iterating through image objects.
            image_path = image_object.get("path")  # Fetching location of image.
            try:
                image = Image.open(image_path).convert("RGB")  # Converting to RGB if not already.
            except Exception as e:
                print(f"Failed to load {image_path}: {e}")
                continue
            image_tensor = self.preprocess(image).unsqueeze(0).to(self.device)  # Converting into tensor since OpenCLIP can't embed other inputs.
            emb_imag = self.model.encode_image(image_tensor)  # Embedding images
            emb_imag = emb_imag / emb_imag.norm(dim=-1, keepdim=True)  # Normalizing images
            emb_imag = emb_imag.cpu().numpy()[0]  # Converting tensor output to numpy array. Using .cpu since numpy can't access GPU memory.

            caption = image_object.get("caption_text") or ""  # Caption validation
            caption = caption.strip() if isinstance(caption, str) else ""
            if caption:  # If caption is available for the image, then embed it.
                tokens = self.tokenizer([caption]).to(self.device)  # Convert to tensor.
                emb_text = self.model.encode_text(tokens)  # Embedding text or tensor value.
                emb_text = emb_text / emb_text.norm(dim=-1, keepdim=True)  # Normalizing values
                emb_text = emb_text.cpu().numpy()[0]  # Converting tensor to numpy array.
                fused = 0.5 * emb_imag + 0.5 * emb_text  # Fusing image and caption together (more weight to image for PDF visuals).
                norm = np.linalg.norm(fused)
                if norm > 0:
                    fused = fused / norm  # Normalizing fused values.
                image_objects_embeddings.append(fused)  # Appending embeddings.
            else:
                image_objects_embeddings.append(emb_imag)  # If no captions, only images are embedded and appended.

        total_time = perf_counter() - start_time  # Calculate total time took to embedd .
        avg_time = total_time / len(image_objects_embeddings) if image_objects_embeddings else 0  # Calculate average time .

        if not image_objects_embeddings:  # Checking if image embeddings are empty.
            raise ValueError("No valid images were processed. All images failed to load.")

        # Store stats
        stats = {
            "count": len(image_objects),
            "embeddings_created": len(image_objects_embeddings),
            "total_time": total_time,
            "avg_time": avg_time,
            "dimension": open_clip.get_model_config("ViT-L-14")['embed_dim']
        }

        return np.vstack(image_objects_embeddings), total_time, stats  # Returning all the embeddings, time, and stats.

    @torch.no_grad()
    def embed_query(self, query: str) -> np.ndarray:
        if not isinstance(query, str):  # Checking if query is valid.
            raise TypeError("Query must be a string.")
        if not query.strip():
            raise ValueError("Please give a valid prompt.")
        tokens = self.tokenizer([query]).to(self.device)  # Convert to tensor.
        query_embedding = self.model.encode_text(tokens)  # Embedding text or tensor value.
        query_embedding = query_embedding / query_embedding.norm(dim=-1, keepdim=True)  # Normalizing values
        query_embedding = query_embedding.cpu().numpy()[0]  # Converting tensor to numpy array.
        return query_embedding  # Returning the embedding.

In [20]:
from sentence_transformers import SentenceTransformer # Used for loading model .
import numpy as np # Used to store embedding .
import torch # Used for device selection and model execution .
from typing import List # Used for return type .
from langchain_core.documents import Document # Used for storing documents .
from time import perf_counter

In [21]:
# Used for loading model and embedding text .
class TextEmbeddingModel:
    def __init__(self, model_name: str = "BAAI/bge-large-en-v1.5"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"  # Device check.

        try:
            self.model = SentenceTransformer(model_name_or_path=model_name, device=self.device)  # Loading model.
        except Exception as e:
            raise RuntimeError(f"Failed to load SentenceTransformer model: {e}")

        self.query_prefix = "Represent this sentence for searching relevant passages: "

        if self.device == "cuda":
            print(f"BGE running on {torch.cuda.get_device_name(0)}.")
        else:
            print("BGE running on CPU.")
        print(f"Embedding dimension of {model_name} is {self.model.get_sentence_embedding_dimension()}")

    @torch.no_grad()
    def embed_documents(self, documents: List[Document]) -> Tuple:
        if not documents:
            raise ValueError("No documents to embed.")

        texts = []
        for doc in documents:  # Checking if data is available in documents.
            if hasattr(doc, "page_content") and doc.page_content:
                content = doc.page_content.strip()
                if content:
                    texts.append(content)

        if not texts:
            raise ValueError("No valid document content to embed. All documents are empty.")

        # Embedding texts.
        # sentences -> input/texts
        # batch_size -> Number of inputs embedding at a single time.
        # convert_to_numpy -> Convert output to numpy array.
        # normalize_embeddings -> Normalizing all the output vectors.

        start_time = perf_counter()  # Start the timer before text embedding starts . # M1 (Embedding time )

        text_embeddings = self.model.encode(
            sentences=texts,
            batch_size=32,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )
        total_time = perf_counter() - start_time  # Calculate total time took to embedd .
        avg_time = total_time / len(texts)  # Calculate average time .

        # Store stats in dict instead of printing
        stats = {
            "count": len(texts),
            "embeddings_created": len(text_embeddings),
            "total_time": total_time,
            "avg_time": avg_time,
            "dimension": self.model.get_sentence_embedding_dimension()
        }

        return text_embeddings, total_time, stats  # Return embeddings, time, and stats.

    # The following function is used to create embedding for user input prompt.
    @torch.no_grad()
    def embed_query(self, query: str) -> np.ndarray:
        if not isinstance(query, str):  # Checking if query is valid.
            raise TypeError("Query must be a string.")

        if not query.strip():  # Checking if the prompt is not empty.
            raise ValueError("Given prompt is not valid.")

        query = self.query_prefix + query  # Adding a prefix to prompt since BGE is instruction-based (only for query, not documents).

        query_embedding = self.model.encode(  # Embedding query.
            sentences=query,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )

        return query_embedding  # Returning embedded values.

# VectorStore

In [22]:
import json # Used for receiving image object or document while creating a hash id .
from typing import List,Dict,Union,Optional # For datatype of a variable .
import chromadb # Used for creating a vectorDB .
import os # For getting directory path for storing vector database .
import hashlib # Used for creating hashid for documents
import numpy as np
from langchain_core.documents import Document

In [23]:
# The following function is used to create hashid using content of a document or image object .
def stable_hash(obj:dict|str)->str:
    if isinstance(obj,dict):
        obj=json.dumps(obj,sort_keys=True,ensure_ascii=False)
    elif not isinstance(obj,str):
        raise TypeError(f"stable_hash expects dict or str, got {type(obj)}")

    return hashlib.sha256(obj.encode("utf-8")).hexdigest()

In [24]:
# The following function is used to remove None type and replace it with empty string since chromadb cannot store type None .
def sanitize_metadata(metadata: dict) -> dict:
    if not isinstance(metadata,dict): # Type validation .
        raise TypeError(f"sanitize_metadata expects dict , got {type(metadata)}")

    clean = {}
    for k, v in metadata.items():
        if v is None:
            clean[k] = "" # None -> Empty string
        elif isinstance(v, (str, int, float, bool)):
            clean[k] = v # Keep it as it is .
        else:
            clean[k] = str(v) # Convert unknown type to string .

    return clean # Return sanitized metadata .

In [25]:
# Used for initializing vectorDB and also store data in collection .
class VectorStore:
    def __init__(self, collection_name: str, directory: str = "../data/database", silent: bool = False):
        if not collection_name or not isinstance(collection_name, str):
            raise ValueError("Collection name must be a non-empty string.")
        self.collection_name = collection_name  # Collection name.
        self.persistent_directory = directory  # Directory to store database.
        self.collection = None  # Collection, used to store data.
        self.client = None  # Used to connect database.
        self.silent = silent  # Flag to suppress printing
        self.initialize_store()  # Initializing vectorDB.

    def initialize_store(self):
        try:
            os.makedirs(name=self.persistent_directory, exist_ok=True)  # Checking if directory exists; if not, creating one.
            self.client = chromadb.PersistentClient(path=self.persistent_directory)
            if self.collection_exists(self.collection_name):  # Checking if the collection exists; if so, load it.
                if not self.silent:
                    print(f"Loading collection {self.collection_name} from database.")
                self.collection = self.client.get_collection(self.collection_name)
            else:  # If collection does not exist, create it.
                if not self.silent:
                    print(f"New collection {self.collection_name} created in database.")
                self.collection = self.client.create_collection(name=self.collection_name, metadata={"hnsw:space": "cosine"})
            if not self.silent:
                print(f"Vector store initialized.")  # Success message.
                print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:  # Exception handling.
            raise RuntimeError(f"Could not initialize vector store: {e}") from e

    # The following function is used to add data and its embeddings to a collection.
    def add_documents(self, documents: List[Union[Dict, Document]], embeddings: np.ndarray):
        if not self.collection:  # Checking if collection is initialized.
            raise RuntimeError("Collection is not initialized.")
        if not documents:
            raise ValueError("Documents list is empty.")
        if len(documents) != len(embeddings):  # Checking if number of documents and embeddings are the same.
            raise ValueError(f"Number of documents ({len(documents)}) does not match embeddings ({len(embeddings)}).")

        ids, metadatas, texts = [], [], []  # Used to store main content and metadata.
        for doc in documents:
            if isinstance(doc, Document):  # For text embeddings. Document type.
                content = doc.page_content.strip()
                if not content:
                    continue
                metadata = doc.metadata or {}
                metadata = sanitize_metadata(metadata)
                hash_input = {  # Data used for creating hashid.
                    "content": content,
                    "source": metadata.get("source"),
                    "page": metadata.get("page_num", "")
                }
                doc_id = stable_hash(hash_input)
                texts.append(content)  # Appending data to store in vectorDB
                metadatas.append(metadata)
                ids.append(doc_id)
            elif isinstance(doc, Dict):  # For image embeddings. Dict type.
                bbox = doc.get("bbox")
                image_metadata = {
                    "image_path": doc.get("path", ""),
                    "caption_text": doc.get("caption_text", ""),
                    "bbox": json.dumps(bbox) if bbox is not None else "",
                }
                image_metadata = sanitize_metadata(image_metadata)  # Removing None or unknown datatype.
                hash_input = {  # Used for hashid.
                    "image_path": image_metadata["image_path"],
                    "bbox": image_metadata["bbox"],
                    "caption": image_metadata["caption_text"],
                }
                doc_id = stable_hash(hash_input)
                texts.append(doc.get("caption_text", ""))  # Appending data to store in vectorDB
                metadatas.append(image_metadata)
                ids.append(doc_id)
            else:
                raise TypeError(f"Unsupported document type: {type(doc)}")  # If input is neither Document nor Dict.

        if not ids:
            if not self.silent:
                print("No valid documents to process.")
            return

        existing_ids = set(  # Used to check if the document was previously added.
            self.collection.get(include=[])["ids"]
        )
        seen = set()
        new_indices = []
        for i, doc_id in enumerate(ids):  # Check for redundant documents.
            if doc_id in existing_ids or doc_id in seen:
                continue
            seen.add(doc_id)
            new_indices.append(i)

        if not new_indices:  # Checking if there are new documents to add to collection.
            if not self.silent:
                print("No new documents to add")
            return

        self.collection.add(  # Adding new documents to collection
            ids=[ids[i] for i in new_indices],
            documents=[texts[i] for i in new_indices],
            metadatas=[metadatas[i] for i in new_indices],
            embeddings=[embeddings[i].tolist() for i in new_indices]
        )
        if not self.silent:
            print(f"Added {len(new_indices)} new documents to collection.")

    # The following function is used to check if the collection exists.
    def collection_exists(self, collection_name: str) -> bool:
        collections_in_db = self.client.list_collections()
        return any(col.name == collection_name for col in collections_in_db)

    # The following function is used to query and get relevant documents.
    def query(self, query_embedding: np.ndarray, k: int = 5, where: Optional[Dict] = None):
        if not self.collection:  # Checking if collection is initialized.
            raise RuntimeError("Collection is not initialized.")
        if query_embedding.ndim != 1:  # Checking if query is 1-dimensional.
            raise ValueError("Query embedding must be a 1D vector.")
        if not isinstance(k, int) or k <= 0:
            raise ValueError(f"k must be a positive integer, got {k}.")

        results = self.collection.query(  # Storing results.
            query_embeddings=[query_embedding.tolist()],
            n_results=k,  # Getting top results.
            where=where,  # Acts as a filter.
            include=["documents", "metadatas", "distances"]  # Including the following data.
        )
        return results

    # The following function is used to get stats about the collection.
    def get_collection_stats(self) -> Dict:
        if not self.collection:
            raise RuntimeError("Collection is not initialized.")
        # Stats about collection . M2 (Collection count)
        return {
            "name": self.collection_name,
            "count": self.collection.count(),
            "directory": self.persistent_directory
        }

    # The following function is used to delete current collection.
    def delete_collection(self) -> None:
        if not self.collection:
            raise RuntimeError("Collection is not initialized.")
        self.client.delete_collection(name=self.collection_name)
        self.collection = None
        print(f"Collection {self.collection_name} deleted.")

# Retrieval

In [26]:
from PIL import Image
import os
from typing import List,Dict,Optional
from sentence_transformers import CrossEncoder

In [27]:
# Used to rerank the chunks with relevance .
class Reranker:
    def __init__(self):
        from sentence_transformers import CrossEncoder
        self.model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2")
        print("Reranker initialized")

    def rerank(self, query: str, items: List[Dict], top_k: int = 3) -> List[Dict]:
        if not items:  # Checking if chunks are empty.
            return []

        pairs = [[query, item["text"]] for item in items]
        scores = self.model.predict(pairs)  # Reranking distances.

        ranked = sorted(  # Ranking based on score.
            zip(items, scores),
            key=lambda x: x[1],
            reverse=True  # Higher score is better.
        )

        return [item for item, _ in ranked[:top_k]]  # Returning top k chunks.

# Context formatter

In [28]:
# Used to format retrieved docs from retrieval output .
class ContextFormatter:
    def __init__(self, max_text_chunks: int = 3, max_images: int = 1,
                 text_distance_threshold: float = 0.55, image_distance_threshold: float = 0.65):
        if not isinstance(max_text_chunks, int) or max_text_chunks <= 0:  # Threshold value validation.
            raise ValueError(f"max_text_chunks must be a positive integer, got {max_text_chunks}.")
        if not isinstance(max_images, int) or max_images <= 0:
            raise ValueError(f"max_images must be a positive integer, got {max_images}.")
        if not isinstance(text_distance_threshold, (int, float)) or text_distance_threshold < 0:
            raise ValueError(f"text_distance_threshold must be a non-negative number, got {text_distance_threshold}.")
        if not isinstance(image_distance_threshold, (int, float)) or image_distance_threshold < 0:
            raise ValueError(f"image_distance_threshold must be a non-negative number, got {image_distance_threshold}.")

        self.max_text_chunks = max_text_chunks  # Limiting number of chunks.
        self.max_images = max_images  # Limiting number of max images.
        self.text_distance_threshold = float(text_distance_threshold)  # Threshold for filtering chunks (lowered for quality).
        self.image_distance_threshold = float(image_distance_threshold)  # Threshold for filtering images (lowered for quality).

    # Removing unwanted details from results.
    def _flatten_results(self, results: Dict) -> List[Dict]:
        if not results or not isinstance(results, Dict):  # Checking if data is empty.
            return []
        documents = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]
        distances = results.get("distances", [[]])[0]

        min_length = min(len(documents), len(metadatas), len(distances))
        if min_length == 0:
            return []

        flattened = []
        for i in range(min_length):
            flattened.append({  # Appending required data.
                "text": documents[i],
                "metadata": metadatas[i],
                "distance": float(distances[i])
            })
        return flattened  # Returning flattened data.

    # The following function is used to load image from path.
    def _load_image(self, image_path: str) -> Optional[Image.Image]:
        if not image_path or not isinstance(image_path, str):  # Checking if image path is valid.
            return None
        if not os.path.exists(image_path):
            print(f"Warning: Image path does not exist: {image_path}")
            return None
        try:
            return Image.open(image_path).convert("RGB")  # Returning loaded image.
        except Exception as e:
            print(f"Failed to load image {image_path}: {e}")
            return None

    # The following function filters text chunks based on distance.
    def _select_text_chunks(self, text_results: Dict) -> List[Dict]:
        items = self._flatten_results(text_results)  # Removing unwanted data from results.
        if not items:  # Checking if items is empty.
            return []
        filtered = [
            i for i in items if i["distance"] <= self.text_distance_threshold  # Filtering based on distance.
        ]
        if not filtered:  # If no chunks (all distances greater than threshold), return top retrieved.
            items.sort(key=lambda x: x["distance"])
            return items[:self.max_text_chunks]  # Returning max number of chunks.
        filtered.sort(key=lambda x: x["distance"])
        return filtered[:self.max_text_chunks]  # Returning filtered chunks.

    # The following function is to format text in data.
    def _format_text_context(self, text_items: List[Dict]) -> str:
        if not text_items or not isinstance(text_items, list):  # Input validation.
            return ""
        lines = []
        for idx, item in enumerate(text_items, start=1):  # Fetching required data.
            if not isinstance(item, dict):
                continue
            meta = item["metadata"] or {}
            source = meta.get("source", "unknown")
            page = meta.get("page_num", "N/A")
            text = item["text"].strip()
            if not text:
                continue
            lines.append(
                f"[{idx}] {text}\n"
                f"(Source: {source}, page {page})"  # Attaching source and page number.
            )
        return "\n\n".join(lines)  # Returning formatted text.

    # The following function selects images from retrieved output.
    def _select_images(self, image_results: Dict) -> List[Dict]:
        items = self._flatten_results(image_results)  # Removing unwanted data from results.
        if not items:
            return []
        items = [
            i for i in items if i["distance"] <= self.image_distance_threshold  # Filtering based on distance.
        ]
        items.sort(key=lambda x: x["distance"])
        return items[:self.max_images]  # Returning filtered output.

    # The following function is used to format image caption.
    def _format_image_context(self, image_items: List[Dict]) -> List[Dict]:
        if not image_items or not isinstance(image_items, list):
            return []
        formatted_images = []
        for item in image_items:  # Fetching required data.
            if not isinstance(item, dict):
                continue
            meta = item.get("metadata") or {}
            image_path = meta.get("image_path")
            caption = meta.get("caption_text", "").strip()
            image = self._load_image(image_path)  # Loading image.
            if image is None:  # Checking if image is not empty.
                continue
            formatted_images.append({  # Attaching image and its caption.
                "image": image,
                "caption": caption
            })
        return formatted_images  # Returning formatted images.

    # The following function is to format both text and images.
    def format(self, retrieval_output: Dict) -> Dict:
        if not retrieval_output or not isinstance(retrieval_output, dict):
            raise ValueError("retrieval_output must be a non-empty dictionary.")
        query = retrieval_output.get("query", "")  # Fetching query.
        text_items = self._select_text_chunks(
            retrieval_output.get("text_results", {})  # Selecting text chunks.
        )
        image_items = self._select_images(
            retrieval_output.get("image_results", {})  # Selecting images.
        )
        return {  # Returning filtered text and images.
            "query": query,
            "text_context": self._format_text_context(text_items),
            "images": self._format_image_context(image_items)
        }

In [29]:
# Used to retrieve documents from database using input query from user .
class RetrievalRag:
    def __init__(self, image_embedder: ImageEmbeddingModel, text_embedder: TextEmbeddingModel,
                 image_vectordb: VectorStore, text_vectordb: VectorStore,
                 use_reranker: bool = True, formatter: Optional['ContextFormatter'] = None):
        self.image_embedder = image_embedder  # Image embedding model.
        self.text_embedder = text_embedder  # Text embedding model.
        self.image_vectordb = image_vectordb  # Image database.
        self.text_vectordb = text_vectordb  # Text database.
        self.reranker = Reranker() if use_reranker else None
        self.formatter = formatter or ContextFormatter()

    # The following function is used to retrieve text from database.
    def retrieve_text(self, query: str, k: int = 5) -> Dict:
        if not query or not query.strip():  # Query validation.
            raise ValueError("Query must be a non-empty string.")
        if not isinstance(k, int) or k <= 0:
            raise ValueError(f"k must be a positive integer, got {k}.")

        start_total = perf_counter()  # Start time before the retrieval .
        start_embed = perf_counter()
        query_embedding = self.text_embedder.embed_query(query)  # Embedding query.
        embed_time = perf_counter() - start_embed  # Calculate query embedding time .
        start_query = perf_counter()
        results = self.text_vectordb.query(query_embedding=query_embedding, k=k)
        query_time = perf_counter() - start_query  # Calculate retrieval time .
        total_time = perf_counter() - start_total  # Calculate total time .

        results["retrieval_metrics"] = {
            "text_embed_time": round(embed_time, 4),
            "text_search_time": round(query_time, 4),
            "text_total_retrieval_time": round(total_time, 4)
        }
        return results  # Returning top k results.

    # The following function is used to retrieve images and their captions from database.
    def retrieve_images(self, query: str, k: int = 3) -> Dict:
        if not query or not query.strip():  # Query validation.
            raise ValueError("Query must be a non-empty string.")
        if not isinstance(k, int) or k <= 0:
            raise ValueError(f"k must be a positive integer, got {k}.")

        start_total = perf_counter()  # Start time before the retrieval .
        start_embed = perf_counter()
        query_embedding = self.image_embedder.embed_query(query)  # Embedding query.
        embed_time = perf_counter() - start_embed  # Calculate query embedding time .
        start_query = perf_counter()
        results = self.image_vectordb.query(
            query_embedding=query_embedding,
            k=k
        )
        query_time = perf_counter() - start_query  # Calculate retrieval time .
        total_time = perf_counter() - start_total  # Calculate total time .

        results["retrieval_metrics"] = {
            "image_embed_time": round(embed_time, 4),
            "image_search_time": round(query_time, 4),
            "image_total_retrieval_time": round(total_time, 4)
        }
        return results

    # Used to retrieve multimodal data.
    def retrieve(self, query: str, text_k: int = 10, image_k: int = 5, rerank_k: int = 5) -> Dict:
        if not query or not query.strip():  # Checking if query is valid.
            raise ValueError("Query must be non-empty string.")
        if not isinstance(text_k, int) or text_k <= 0:  # Threshold validation.
            raise ValueError(f"text_k must be a positive integer, got {text_k}.")
        if not isinstance(image_k, int) or image_k <= 0:
            raise ValueError(f"image_k must be a positive integer, got {image_k}.")

        start_total = perf_counter()  # Start time before retrieval of images and text .
        # M3 (Retrieval time)
        text_results = self.retrieve_text(query=query, k=text_k)
        image_results = self.retrieve_images(query=query, k=image_k)

        # For quality: Boost text chunk scores if they have related images retrieved.
        text_items = self.formatter._flatten_results(text_results)
        image_ids = [i["metadata"].get("image_path", "").split('/')[-1]
                     for i in self.formatter._flatten_results(image_results)]

        # Extract retrieved image IDs.
        for item in text_items:
            related_ids = item["metadata"].get("related_image_ids", [])
            overlap = set(related_ids) & set(image_ids)
            if overlap:
                item["distance"] *= 0.9  # Boost (lower distance) if overlap for hybrid quality.

        if self.reranker:  # Checking if reranking is initialized.
            reranked = self.reranker.rerank(
                query=query,
                items=text_items,
                top_k=min(rerank_k, len(text_items)),
            )
            # Rebuild results in original structure.
            text_results["documents"] = [[i["text"] for i in reranked]]
            text_results["metadatas"] = [[i["metadata"] for i in reranked]]
            text_results["distances"] = [[i["distance"] for i in reranked]]

        overall_time = perf_counter() - start_total  # Calculate overall time .

        # ───────────── M4 Cosine Similarity calculation ───────────────────────
        distances = []
        if "distances" in text_results and text_results["distances"]:
            distances.extend(text_results["distances"][0])
        if "distances" in image_results and image_results["distances"]:
            distances.extend(image_results["distances"][0])
        cosine_sim = sum(1 - d for d in distances) / len(distances) if distances else 0.0

        # Overall retrieval time stats .
        combined_metrics = {
            "overall_retrieval_time": overall_time,
            "cosine_similarity": cosine_sim,
            "text_metrics": text_results.get("retrieval_metrics"),
            "image_metrics": image_results.get("retrieval_metrics")
        }

        return {  # Returning retrieved documents from database.
            "query": query,
            "text_results": text_results,
            "image_results": image_results,
            "retrieval_metrics": combined_metrics
        }

In [30]:
from typing import List, Dict
import matplotlib.pyplot as plt
from PIL import Image

# Testing

In [31]:
TEST_QUESTIONS = [
    # ======================
    # SECTION 1: VOYAGER GRAND TOUR (Questions 1-8)
    # ======================
    {
        "id": 1,
        "type": "temporal_trajectory_reasoning",
        "question": "Why does the visual geometry prove that gravity assists fundamentally redirect trajectories rather than merely accelerate the spacecraft, and what irreversible decision did this force upon Voyager 1 at Saturn?",
        "expected_images": True,
        "image_reference": {"page": 1, "position": "Left diagram showing flight paths of Voyager 1 and 2 through outer solar system"},
        "required_context": {"pages": [1], "focus": "Introduction and flight trajectories explanation including gravity assists and Titan flyby trade-off"}
    },
    {
        "id": 2,
        "type": "counterintuitive_scale_perspective",
        "question": "Why was this photograph considered a major engineering milestone even though the spacecraft was still relatively close to Earth by solar system standards?",
        "expected_images": True,
        "image_reference": {"page": 2, "position": "Left image: First photograph of Earth-Moon system in single frame"},
        "required_context": {"pages": [2], "focus": "Text describing the first Earth-Moon photo and its distance of 7.25 million miles"}
    },
    {
        "id": 3,
        "type": "visual_evidence_vs_claim",
        "question": "What specific visual features in this montage support the claims of active volcanoes on Io and a possible subsurface ocean on Europa?",
        "expected_images": True,
        "image_reference": {"page": 3, "position": "Left montage: Jupiter and its four largest moons (not to scale)"},
        "required_context": {"pages": [3], "focus": "Discoveries at Jupiter including Io volcanoes, Europa ice surface, and new small moons"}
    },
    {
        "id": 4,
        "type": "philosophical_visual_synthesis",
        "question": "Why did the mission team deliberately wait until just before permanently shutting down the cameras to capture this image, and what deeper message does it convey?",
        "expected_images": True,
        "image_reference": {"page": 4, "position": "Bottom image: Family Portrait of six planets from 6 billion km"},
        "required_context": {"pages": [4], "focus": "Text about the Family Portrait mosaic and camera shutdown for power conservation"}
    },
    {
        "id": 5,
        "type": "artifact_inference",
        "question": "Why was 1970s analog technology deliberately chosen for a message that might be found tens of thousands of years in the future?",
        "expected_images": True,
        "image_reference": {"page": 5, "position": "Both images: Golden Record with cover and contents"},
        "required_context": {"pages": [5], "focus": "Description of Golden Record contents, playback instructions, and stylus"}
    },
    # {
    #     "id": 6,
    #     "type": "trajectory_decision_reasoning",
    #     "question": "What scientific trade-off led to Voyager 1's flyby permanently sacrificing all future planetary encounters while Voyager 2 was allowed to continue?",
    #     "expected_images": True,
    #     "image_reference": {"page": 3, "position": "Right montage: Saturn and several of its moons (not to scale)"},
    #     "required_context": {"pages": [3], "focus": "Discoveries at Saturn including Titan lakes, rings complexity, and new satellites"}
    # },
    # {
    #     "id": 7,
    #     "type": "discovery_density_reasoning",
    #     "question": "Why is it counterintuitive that Voyager 2 discovered more new moons at these distant ice giants than at Jupiter and Saturn combined, despite spending far less time observing them?",
    #     "expected_images": True,
    #     "image_reference": {"page": 4, "position": "Left and right montages: Uranus with moons and Neptune with Triton"},
    #     "required_context": {"pages": [4], "focus": "Discoveries at Uranus and Neptune including new moons, magnetic fields, and Triton geysers"}
    # },
    # {
    #     "id": 8,
    #     "type": "long_duration_philosophy",
    #     "question": "How does the visible design explain why both spacecraft could still be operating after 45+ years when most missions are planned for only a few years?",
    #     "expected_images": True,
    #     "image_reference": {"page": 2, "position": "Right image: Voyager spacecraft as it appears in space"},
    #     "required_context": {"pages": [2, 5], "focus": "Spacecraft description and interstellar mission with power projections to 2025"}
    # },
    #
    # # ======================
    # # SECTION 2: CURIOSITY / MARS SCIENCE LABORATORY (Questions 9-16)
    # # ======================
    # {
    #     "id": 9,
    #     "type": "geological_narrative_reconstruction",
    #     "question": "Reconstruct the full ancient river system story and explain why this image alone was not sufficient to prove past habitability.",
    #     "expected_images": True,
    #     "image_reference": {"page": 2, "position": "Top-left image: Rock outcrop called Link with rounded pebbles"},
    #     "required_context": {"pages": [2], "focus": "Text about evidence for stream flow and conglomerate rocks"}
    # },
    # {
    #     "id": 10,
    #     "type": "engineering_precision_causality",
    #     "question": "Why was the 20 km landing precision improvement not just an engineering achievement but a fundamental prerequisite for the entire scientific mission?",
    #     "expected_images": True,
    #     "image_reference": {"page": 1, "position": "Top-right image: Curiosity lowered by sky crane"},
    #     "required_context": {"pages": [1], "focus": "Mission Overview including landing technology and site selection"}
    # },
    # {
    #     "id": 11,
    #     "type": "self_portrait_technical_reasoning",
    #     "question": "How was this image technically assembled, and what does it reveal about the rover's mobility and self-diagnostic capabilities?",
    #     "expected_images": True,
    #     "image_reference": {"page": 3, "position": "Top-left image: Curiosity self-portrait"},
    #     "required_context": {"pages": [3], "focus": "Text about rover size and design inheritance from Spirit/Opportunity"}
    # },
    # {
    #     "id": 12,
    #     "type": "drill_evidence_inference",
    #     "question": "How does the powdered interior sample, combined with the 4.2 billion year age, create a paradox about geological preservation on Mars?",
    #     "expected_images": True,
    #     "image_reference": {"page": 2, "position": "Bottom-right image: First sample drilling at John Klein"},
    #     "required_context": {"pages": [2], "focus": "Text about drilled sample analysis and rock age determination"}
    # },
    # {
    #     "id": 13,
    #     "type": "power_system_counterintuitive",
    #     "question": "Why was the documented decline in radioisotope power from 110W to 100W over two years actually a positive indicator for long-term mission success?",
    #     "expected_images": True,
    #     "image_reference": {"page": 3, "position": "Center image: Big rover illustration or context"},
    #     "required_context": {"pages": [3], "focus": "Text about radioisotope power system and decline"}
    # },
    # {
    #     "id": 14,
    #     "type": "habitability_hierarchy",
    #     "question": "Why does the document argue that visual imaging alone could never have proven past habitability, and what hierarchical chain of instruments was required?",
    #     "expected_images": True,
    #     "image_reference": {"page": 3, "position": "Center image: Rover with Mast Camera and ChemCam"},
    #     "required_context": {"pages": [3], "focus": "Text about science payload and instruments"}
    # },
    # {
    #     "id": 15,
    #     "type": "landing_site_synthesis",
    #     "question": "Why was Gale Crater chosen over dozens of other sites, and how did the payload directly address the habitability question?",
    #     "expected_images": True,
    #     "image_reference": {"page": 1, "position": "Bottom image: Gale Crater size comparison"},
    #     "required_context": {"pages": [1], "focus": "Text about landing site and crater size"}
    # },
    # {
    #     "id": 16,
    #     "type": "atmospheric_loss_mechanism",
    #     "question": "Explain the counterintuitive mechanism by which Mars lost most of its atmosphere from the top rather than the surface.",
    #     "expected_images": True,
    #     "image_reference": {"page": 2, "position": "Top-left image: Link outcrop"},
    #     "required_context": {"pages": [2], "focus": "Text about atmospheric composition analysis and loss process"}
    # },
    #
    # # ======================
    # # SECTION 3: NPP BROCHURE (Questions 17-22)
    # # ======================
    # {
    #     "id": 17,
    #     "type": "disaster_visual_reasoning",
    #     "question": "How does this single satellite image distinguish between active fire, burned area, and smoke plume?",
    #     "expected_images": True,
    #     "image_reference": {"page": 5, "position": "Bottom image: Wallow Fire satellite view"},
    #     "required_context": {"pages": [5], "focus": "Text about disaster monitoring and fire detection"}
    # },
    # {
    #     "id": 18,
    #     "type": "vegetation_trend_analysis",
    #     "question": "What hemispheric asymmetry does this image reveal about climate change impacts, and why was extending the MODIS record with VIIRS scientifically essential?",
    #     "expected_images": True,
    #     "image_reference": {"page": 4, "position": "Center map: Vegetation productivity global view"},
    #     "required_context": {"pages": [4], "focus": "Text about vegetation trends and pine beetle damage"}
    # },
    # {
    #     "id": 19,
    #     "type": "radiation_budget_reasoning",
    #     "question": "Why is measuring both reflected sunlight and emitted heat essential for understanding climate feedbacks, and how do clouds complicate this measurement?",
    #     "expected_images": True,
    #     "image_reference": {"page": 7, "position": "Center image: CERES instrument or radiation budget diagram"},
    #     "required_context": {"pages": [7], "focus": "Text about CERES and radiation budget"}
    # },
    # {
    #     "id": 20,
    #     "type": "day_night_advance",
    #     "question": "Why does the ability to detect moonlight-illuminated clouds at night represent a major advance over previous imagers?",
    #     "expected_images": True,
    #     "image_reference": {"page": 6, "position": "Center image: Day/night band example"},
    #     "required_context": {"pages": [6], "focus": "Text about VIIRS day/night band capabilities"}
    # },
    # {
    #     "id": 21,
    #     "type": "bridge_mission_philosophy",
    #     "question": "Why was NPP designed as a 'bridge' mission, and what specific data continuity risks did it mitigate?",
    #     "expected_images": True,
    #     "image_reference": {"page": 1, "position": "Center image: NPP satellite over Earth"},
    #     "required_context": {"pages": [1], "focus": "Text about NPP as bridge to new era of observations"}
    # },
    # {
    #     "id": 22,
    #     "type": "climate_cascade_effects",
    #     "question": "How does a temperature increase of just a few degrees trigger cascading ecological effects visible from space?",
    #     "expected_images": True,
    #     "image_reference": {"page": 4, "position": "Bottom-left image: Pine beetle damage"},
    #     "required_context": {"pages": [4], "focus": "Text about climate system changes and beetle infestation"}
    # },
    #
    # # ======================
    # # SECTION 4: CROSS-DOCUMENT SYNTHESIS (Questions 23-25)
    # # ======================
    # {
    #     "id": 23,
    #     "type": "cross_mission_scale_perspective",
    #     "question": "What fundamentally different philosophical statements do these two images from opposite directions make about humanity's place in the universe?",
    #     "expected_images": True,
    #     "image_reference": {"page": 4, "position": "Bottom image: Voyager Family Portrait"},
    #     "required_context": {"pages": [4, 1], "focus": "Family Portrait text (Voyager) + NPP satellite overview"}
    # },
    # {
    #     "id": 24,
    #     "type": "cross_mission_habitability_method",
    #     "question": "Why did proving past habitability on Mars require physical sampling while inferring possible present habitability on Europa relied primarily on remote visual evidence?",
    #     "expected_images": True,
    #     "image_reference": {"page": 3, "position": "Left montage: Jupiter and moons (Voyager)"},
    #     "required_context": {"pages": [3, 2], "focus": "Europa ocean text (Voyager) + John Klein drill analysis (Curiosity)"}
    # },
    # {
    #     "id": 25,
    #     "type": "long_duration_design_philosophy",
    #     "question": "What do these three artifacts reveal about different philosophies of designing for extremely long-duration exploration?",
    #     "expected_images": True,
    #     "image_reference": {"page": 5, "position": "Both images: Golden Record (Voyager)"},
    #     "required_context": {"pages": [5, 3, 1], "focus": "Golden Record (Voyager) + RTG power (Curiosity) + NPP data continuity"}
    # }
]

# LLM

In [32]:
import ollama # Used to load model .
from textwrap import dedent # Used for spacing problems in prompt .
from tabulate import tabulate # Used for creating a table for displaying models .
import subprocess # Used to start ollama server .
import time # For waiting .
import requests # Used to access ollama server .
import base64
import io
from PIL import Image
from sentence_transformers import util
import re

In [33]:
# Used to initialize a language model and generate responses .
class LocalLLM:
    def __init__(self, model_name: str = "gemma3:4b", text_embedder: TextEmbeddingModel = None):
        # Default model; can change.
        self.model_name = model_name
        self.process = self.ollama_server(process="start")
        self.text_embedder = text_embedder or TextEmbeddingModel()
        if not self.is_model_available(self.model_name):
            # Checking if model is available or valid.
            available = [m['model_name'] for m in self.available_models()]
            raise ValueError(
                f"Model '{model_name}' not available. "
                f"Available models: {available}"
            )
        self.tokenizer = tiktoken.get_encoding("cl100k_base")

    # The following function is used to start or stop ollama server.
    def ollama_server(self, process: str):
        if process == "start":
            try:
                requests.get("http://localhost:11434/api/tags", timeout=1)  # Checking if ollama server is already started.
                print("Ollama already running")
                return "external"
            except:
                pass
            ollama_process = subprocess.Popen(  # Starting ollama server if not started.
                ["ollama", "serve"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
                shell=False
            )
            for _ in range(10):  # Checking if server started.
                try:
                    requests.get("http://localhost:11434/api/tags", timeout=1)
                    print("Ollama server started")
                    return ollama_process
                except:
                    time.sleep(1)
            raise RuntimeError("Ollama failed to start")  # If server did not start after many tries, raise error.

        elif process == "stop":  # Stopping ollama server.
            if isinstance(self.process, subprocess.Popen):  # Checking if server was started here.
                self.process.terminate()
                self.process.wait()
                print("Ollama server successfully stopped.")
            else:
                print("Ollama was not started by this process")  # If started externally, notify.
            return None
        else:
            raise ValueError("Input can be either 'start' or 'stop'")  # Input validation.

    # The following function is used to check available models in the local device.
    def available_models(self):
        response = ollama.list()  # Getting available models.
        models_available = response.get('models', [])  # Safer access
        models = []
        for m in models_available:  # Getting required information from model.
            models.append({
                "model_name": m.get('model'),
                "parameters": m.get('details', {}).get('parameter_size')
            })
        return models if models else []

    # The following function is to check if a specific model is available in local device.
    def is_model_available(self, model_name):
        models = self.available_models()
        return any(m['model_name'] == model_name for m in models)

    # The following function is used to build prompt using user query and retrieved documents.
    # Enhanced prompt with chain-of-thought reasoning for better quality answers.
    def build_prompt(self, query: str, context: str):
        return f"""You are a rigorous scientific analyst specialized in PDF documents with images and captions. Your task is to answer the question using ONLY the provided context, images, and captions. You must remain strictly evidence-based and avoid speculation. You are not allowed to use external knowledge, assumptions, or inferred facts. If the available evidence is insufficient, you must clearly explain why.

----------------------------------------------------------------
CORE PRINCIPLES
----------------------------------------------------------------
- Use only the provided text, images, and captions.
- Prioritize visual evidence from images if they directly relate to the text.
- Do not introduce outside knowledge.
- Do not assume missing details.
- If the text does not explicitly support the answer, clearly state that the context is insufficient.
- If an image or caption contradicts the text, clearly explain the inconsistency.
- If an image is unrelated to the object described in the text, explicitly state that it is not relevant.
- Before evaluating relevance, verify that the image depicts the SAME object or phenomenon mentioned in the text.
- Emphasize captions as they provide direct context to images.

----------------------------------------------------------------
REQUIRED STRUCTURE - USE CHAIN-OF-THOUGHT REASONING
----------------------------------------------------------------
1) Textual Evidence Assessment
   - Identify the specific object(s), phenomenon, or event described in the text.
   - Determine whether the text explicitly supports the question.
   - Summarize the exact supporting statements.
   - If the text does not adequately support the answer, explain why and stop.

2) Image and Caption Evaluation
   - Images provided: Yes / No
   - For each image: Identify what object or phenomenon is shown, describe the caption if present.
   - Compare it to the object described in the text.
   - State whether they refer to the same object.
   - If they refer to different objects, clearly state that the image is not relevant.
   - Describe only what is directly visible.
   - Conclude whether the image/caption:
     • Supports the text
     • Contradicts the text
     • Is unrelated or insufficient

3) Integrated Reasoning with Chain-of-Thought
   - Think step-by-step about how the evidence connects to the question.
   - Connect the validated textual evidence with any relevant visual/caption evidence.
   - Explain mechanisms, processes, and any numerical details mentioned.
   - Identify logical steps that link evidence to conclusion.
   - Show your reasoning process explicitly.
   - Explicitly mention any limitations or missing information.

4) Final Conclusion
   - Provide a well-structured, natural explanation based on your reasoning.
   - Minimum 8–12 detailed sentences.
   - The conclusion must strictly follow from validated evidence.
   - Do not introduce any information not present in the provided material.
   - Cite specific sources when making claims (e.g., "According to [Source], page [X]...").

----------------------------------------------------------------
Context: {context}

Question: {query}

Answer (think step-by-step):""".strip()

    # The following function is used to convert PIL image to base64 since LLM models can read base64 or need image path.
    @staticmethod
    def pil_to_base64(img: Image.Image) -> str:
        buffer = io.BytesIO()
        img = img.convert("RGB")
        img.save(buffer, format="PNG")
        return base64.b64encode(buffer.getvalue()).decode("utf-8")

    # The following function is used to generate response from language model.
    def generate_response(self, query: str, context: str, images: Optional[List[Dict]] = None,
                         stream: bool = True, temperature: float = 0.7, max_tokens: int = 500) -> Dict:
        if not query or not query.strip():  # Query validation.
            raise ValueError("Query cannot be empty")
        if not context or not context.strip():
            if not images:
                raise ValueError("Context cannot be empty when no images are provided")  # Context validation.
        if len(context) > 10000:  # Checking if context is too large.
            print("Warning: Large context may be slow")

        try:
            prompt = self.build_prompt(query, context)  # Building a prompt using query and context.
            image_payload = []
            if images:
                for img_dict in images:
                    img = img_dict.get("image")
                    if isinstance(img, Image.Image):
                        image_payload.append(self.pil_to_base64(img))
                    else:
                        raise TypeError("Images must be PIL.Image.Image")

            # Context length . M9 (Input context length)
            context_chars = len(context)
            context_tokens = len(self.tokenizer.encode(context))

            start_time = time.perf_counter()
            response = ollama.chat(  # Getting response from model.
                model=self.model_name,
                messages=[
                    {"role": "system", "content": "You are a grounded assistant that answers only from provided text and images. Think step-by-step and show your reasoning."},
                    {
                        "role": "user",
                        "content": prompt,
                        "images": image_payload if image_payload else None
                    }
                ],
                stream=stream,
                options={
                    'temperature': temperature,
                    "num_predict": max_tokens
                },
                keep_alive=0
            )

            if stream:  # Displaying output through streaming.
                print(f"\n{'='*80}")
                print(f"QUERY: {query}")
                print(f"{'='*80}")
                print("ANSWER:")
                print("-" * 80)
                full_response = ""
                try:
                    for chunk in response:  # Displaying response as model gives output.
                        content = chunk.get("message", {}).get("content", "")
                        if content:
                            print(content, end="", flush=True)
                            full_response += content
                    print("\n" + "=" * 80)
                except Exception as e:
                    print(f"\nError during streaming: {e}")
                    raise
                final_response = full_response
            else:
                final_response = response["message"]["content"]  # If stream is off, give output all at once.

            generation_time = time.perf_counter() - start_time

            # ──────────────────── M13: Factual Consistency Distance (FCD) ───────────────────────────────
            fcd = None
            try:
                resp_emb = self.text_embedder.embed_query(final_response)
                ctx_emb = self.text_embedder.embed_query(prompt)
                sim = util.cos_sim(resp_emb, ctx_emb).item()
                dist = 1 - sim
                fcd = dist * 100
            except Exception as e:
                print(f"FCD computation failed: {e}")
                fcd = None

            # ────────────── M14: Faithfulness / Citation Recall ───────────────────────────────
            faithfulness = 0.0
            try:
                response_lower = final_response.lower()
                cited_sources = set(re.findall(r'(source:\s*[^,\n]+?|\w+\.pdf|https?://[^\s<>\n]+)', response_lower, re.IGNORECASE))
                cited_pages = set(re.findall(r'page\s*(\d+)', response_lower, re.IGNORECASE))
                cited_images = set(re.findall(r'(image\s*\d+|caption|figure\s*\d+)', response_lower, re.IGNORECASE))

                context_lower = context.lower()
                retrieved_sources = set(re.findall(r'(source:\s*[^,\n]+?|\w+\.pdf|https?://[^\s<>\n]+)', context_lower, re.IGNORECASE))
                retrieved_pages = set(re.findall(r'page\s*(\d+)', context_lower, re.IGNORECASE))
                retrieved_images = set(re.findall(r'(image\s*\d+|caption|figure\s*\d+)', context_lower, re.IGNORECASE))

                total_retrieved = len(retrieved_sources | retrieved_pages | retrieved_images) or 1
                total_cited = len(cited_sources | cited_pages | cited_images)
                faithfulness = (total_cited / total_retrieved) * 100
            except Exception as e:
                print(f"M14 Faithfulness computation failed: {e}")
                faithfulness = 0.0

            return {
                "response": final_response,
                "context_length_chars": context_chars,
                "context_length_tokens": context_tokens,
                "generation_time_sec": round(generation_time, 4),
                "factual_consistency_distance": round(fcd, 2) if fcd is not None else None,
                "faithfulness_percentage": round(faithfulness, 2)
            }
        except Exception as e:
            raise RuntimeError(f"Error generating response: {e}") from e

In [34]:
import psutil

# METRICS

In [35]:
from rouge_score import rouge_scorer
import statistics

In [36]:
class ResourceMonitor:
    def __init__(self):
        self.process = psutil.Process(os.getpid())
        # Initialize CPU percent to get accurate readings later
        self.process.cpu_percent(interval=None)

    def get_snapshot(self):
        # Get CPU percent with interval=None for non-blocking read (requires previous call)
        cpu_percent = self.process.cpu_percent(interval=None)
        return {
            "cpu_percent": cpu_percent,
            "ram_gb": round(self.process.memory_info().rss / (1024 ** 3), 4)
        }

In [37]:
# M1 (Embedding Time)
def compute_embedding_time(text_time: float, image_time: float) -> float:
    total = text_time + image_time
    print(f"\n{'─'*60}")
    print(f"  M1 Embedding Time: {total:.4f} seconds")
    print(f"{'─'*60}")
    return total

In [38]:
# M2 (Index Size)
def compute_index_size(text_db: 'VectorStore', image_db: 'VectorStore') -> int:
    text_count = text_db.get_collection_stats().get("count", 0)
    image_count = image_db.get_collection_stats().get("count", 0)
    total = text_count + image_count
    print(f"\n{'─'*60}")
    print(f"  M2 Index Size: {total} vectors")
    print(f"     └─ Text vectors: {text_count}")
    print(f"     └─ Image vectors: {image_count}")
    print(f"{'─'*60}")
    return total

In [39]:
# M3 (Retrieval Latency)
def compute_retrieval_latency(times: List[float]) -> float:
    if not times:
        print(f"\n{'─'*60}")
        print(f"  M3 Retrieval Latency: 0.0000 seconds")
        print(f"{'─'*60}")
        return 0.0
    avg = statistics.mean(times)
    print(f"\n{'─'*60}")
    print(f"  M3 Retrieval Latency: {avg:.4f} seconds")
    print(f"     └─ Min: {min(times):.4f}s, Max: {max(times):.4f}s")
    print(f"{'─'*60}")
    return avg

In [40]:
# M4 (Cosine Similarity)
def compute_cosine_similarity(values: List[float]) -> float:
    if not values:
        print(f"\n{'─'*60}")
        print(f"  M4 Cosine Similarity: 0.0000")
        print(f"{'─'*60}")
        return 0.0
    avg = statistics.mean(values)
    print(f"\n{'─'*60}")
    print(f"  M4 Cosine Similarity: {avg:.4f}")
    print(f"{'─'*60}")
    return avg

In [41]:
# M5 (top - k accuracy)
def compute_top_k_accuracy(
    retrieval_output: List[Dict],
    test_questions: List[Dict],
    k: int = 5
) -> float:
    gt_pages_map = {}
    for q in test_questions:
        qid = q.get("id")
        if qid:
            pages = q.get("required_context", {}).get("pages", [])
            if pages:
                gt_pages_map[qid] = set(pages)

    matches = 0
    total = 0
    skipped = 0

    for item in retrieval_output:
        qid = item.get("id")
        if qid not in gt_pages_map:
            skipped += 1
            continue

        correct_pages = gt_pages_map[qid]

        try:
            metas = item["result"]["text_results"]["metadatas"][0][:k]
        except (KeyError, IndexError, TypeError):
            skipped += 1
            continue

        retrieved_pages = {m.get("page_num") or m.get("page") for m in metas if m.get("page_num") is not None or m.get("page") is not None}

        if correct_pages & retrieved_pages:
            matches += 1

        total += 1

    if total == 0:
        print(f"\n{'─'*60}")
        print(f"  M5 Top-k Accuracy: 0.00% (no valid queries)")
        print(f"{'─'*60}")
        return 0.0

    score = (matches / total) * 100
    print(f"\n{'─'*60}")
    print(f"  M5 Top-k Accuracy: {score:.2f}%")
    print(f"     └─ Matches: {matches}/{total} queries")
    print(f"{'─'*60}")
    return score

In [42]:
# M6 (ROUGE-1) - FIXED: Properly use ground truth from test_questions
def compute_rouge1(per_query_results: List[Dict], k: int = 5) -> float:
    """
    M6: ROUGE-1 between generated response and retrieved context (top-k chunks as pseudo-ground-truth).
    Measures unigram overlap between response and retrieved documents.
    """
    scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)
    scores = []
    skipped = 0

    for item in per_query_results:
        response_text = item.get("response_text", "").strip()
        context_text = item.get("context", "").strip()

        if not response_text or not context_text:
            skipped += 1
            continue

        # Use top-k chunks as pseudo ground truth
        context_parts = context_text.split("\n\n")
        top_k_context = "\n\n".join(context_parts[:k]) if len(context_parts) > k else context_text

        try:
            score = scorer.score(top_k_context, response_text)["rouge1"].fmeasure
            scores.append(score)
        except Exception as e:
            print(f"  M6 calculation failed: {e}")
            skipped += 1

    avg_score = statistics.mean(scores) if scores else 0.0

    print(f"\n  {'─'*60}")
    print(f"  M6 ROUGE-1: {avg_score:.4f}")
    print(f"       Evaluated: {len(scores)}, Skipped: {skipped}")
    print(f"{'─'*60}")

    return avg_score

In [43]:
# M7 (ROUGE-2) - FIXED: Properly use ground truth from test_questions
def compute_rouge2(per_query_results: List[Dict], k: int = 5) -> float:
    """
    M7: ROUGE-2 between generated response and retrieved context.
    Measures bigram (2-word sequence) overlap - stricter than ROUGE-1.
    """
    scorer = rouge_scorer.RougeScorer(["rouge2"], use_stemmer=True)
    scores = []
    skipped = 0

    for item in per_query_results:
        response_text = item.get("response_text", "").strip()
        context_text = item.get("context", "").strip()

        if not response_text or not context_text:
            skipped += 1
            continue

        # Use top-k chunks as pseudo ground truth
        context_parts = context_text.split("\n\n")
        top_k_context = "\n\n".join(context_parts[:k]) if len(context_parts) > k else context_text

        try:
            score = scorer.score(top_k_context, response_text)["rouge2"].fmeasure
            scores.append(score)
        except Exception as e:
            print(f"  M7 calculation failed: {e}")
            skipped += 1

    avg_score = statistics.mean(scores) if scores else 0.0

    print(f"\n  {'─'*60}")
    print(f"  M7 ROUGE-2: {avg_score:.4f}")
    print(f"       Evaluated: {len(scores)}, Skipped: {skipped}")
    print(f"{'─'*60}")

    return avg_score

In [44]:
# M8 (ROUGE-L)
def compute_rougeL(per_query_results: List[Dict], k: int = 5) -> float:
    """
    M8: ROUGE-L between generated response and retrieved context.
    Measures longest common subsequence - captures sentence structure similarity.
    """
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores = []
    skipped = 0

    for item in per_query_results:
        response_text = item.get("response_text", "").strip()
        context_text = item.get("context", "").strip()

        if not response_text or not context_text:
            skipped += 1
            continue

        # Use top-k chunks as pseudo ground truth
        context_parts = context_text.split("\n\n")
        top_k_context = "\n\n".join(context_parts[:k]) if len(context_parts) > k else context_text

        try:
            score = scorer.score(top_k_context, response_text)["rougeL"].fmeasure
            scores.append(score)
        except Exception as e:
            print(f"  M8 calculation failed: {e}")
            skipped += 1

    avg_score = statistics.mean(scores) if scores else 0.0

    print(f"\n  {'─'*60}")
    print(f"  M8 ROUGE-L: {avg_score:.4f}")
    print(f"       Evaluated: {len(scores)}, Skipped: {skipped}")
    print(f"{'─'*60}")

    return avg_score

In [45]:
# M9 (Context Length) - ADDED: Missing function
def compute_context_length(formatted_output: List[Dict]) -> float:
    """
    Computes average context length in characters from formatted output.
    """
    if not formatted_output:
        print(f"\n{'─'*60}")
        print(f"  M9 Context Length: 0.00 characters")
        print(f"{'─'*60}")
        return 0.0

    lengths = []
    for item in formatted_output:
        text_context = item.get("text_context", "")
        if text_context:
            lengths.append(len(text_context))

    if not lengths:
        print(f"\n{'─'*60}")
        print(f"  M9 Context Length: 0.00 characters")
        print(f"{'─'*60}")
        return 0.0

    avg = statistics.mean(lengths)
    print(f"\n{'─'*60}")
    print(f"  M9 Context Length: {avg:.2f} characters")
    print(f"     └─ Min: {min(lengths)}, Max: {max(lengths)}")
    print(f"{'─'*60}")
    return avg

In [46]:
# M15 (Ground truth coverage) -
def compute_context_coverage(per_query_results: List[Dict]) -> float:
    """
    M15: Context Coverage - percentage of retrieved context terms used in response.
    Measures how well the response utilizes retrieved information (anti-hallucination).
    """
    coverage_scores = []
    skipped = 0

    for item in per_query_results:
        context = item.get("context", "")
        response = item.get("response_text", "")

        if not context or not response:
            skipped += 1
            continue

        # Extract meaningful terms (4+ chars) from context
        context_terms = set(re.findall(r'\b[A-Za-z]{4,}\b', context.lower()))
        response_terms = set(re.findall(r'\b[A-Za-z]{4,}\b', response.lower()))

        if not context_terms:
            skipped += 1
            continue

        # Calculate coverage: what % of context terms appear in response
        overlap = len(context_terms & response_terms)
        coverage = (overlap / len(context_terms)) * 100
        coverage_scores.append(coverage)

    avg_coverage = statistics.mean(coverage_scores) if coverage_scores else 0.0

    print(f"\n  {'─'*60}")
    print(f"  M15 Context Coverage: {avg_coverage:.2f}%")
    print(f"       Evaluated: {len(coverage_scores)}, Skipped: {skipped}")
    print(f"{'─'*60}")

    return avg_coverage

In [47]:
# M16 (Query to response time)
def compute_e2e_latency(per_query_results: List[Dict]) -> float:
    times = [r["e2e_latency_sec"] for r in per_query_results]
    avg = statistics.mean(times) if times else 0.0
    print(f"\n{'─'*60}")
    print(f"  M16 E2E Latency: {avg:.4f} seconds")
    if times:
        print(f"     └─ Min: {min(times):.4f}s, Max: {max(times):.4f}s")
    print(f"{'─'*60}")
    return avg

In [48]:
# M17 (Queries processed per second)
def compute_throughput(per_query_results: List[Dict]) -> float:
    times = [r["e2e_latency_sec"] for r in per_query_results]
    avg_e2e = statistics.mean(times) if times else 0.0
    tp = 1.0 / avg_e2e if avg_e2e > 0 else 0.0
    print(f"\n{'─'*60}")
    print(f"  M17 Throughput: {tp:.3f} queries per second")
    print(f"{'─'*60}")
    return tp

In [49]:
# M18 (CPU usage)
def compute_cpu_usage(per_query_results: List[Dict]) -> float:
    values = [r["avg_cpu_percent"] for r in per_query_results]
    avg = statistics.mean(values) if values else 0.0
    print(f"\n{'─'*60}")
    print(f"  M18 CPU Usage: {avg:.2f}%")
    if values:
        print(f"     └─ Min: {min(values):.1f}%, Max: {max(values):.1f}%")
    print(f"{'─'*60}")
    return avg

In [50]:
# M19 (RAM usage)
def compute_ram_usage(per_query_results: List[Dict]) -> float:
    values = [r["avg_ram_gb"] for r in per_query_results]
    avg = statistics.mean(values) if values else 0.0
    print(f"\n{'─'*60}")
    print(f"  M19 RAM Usage: {avg:.3f} GB")
    if values:
        print(f"     └─ Min: {min(values):.3f}GB, Max: {max(values):.3f}GB")
    print(f"{'─'*60}")
    return avg

In [51]:
# GPU Usage
def compute_gpu_usage(per_query_results: List[Dict]) -> float:
    values = [r.get("avg_gpu_percent", 0.0) for r in per_query_results]
    avg = statistics.mean(values) if values else 0.0
    print(f"\n{'─'*60}")
    print(f"  GPU Usage: {avg:.2f}%")
    if values:
        print(f"     └─ Min: {min(values):.1f}%, Max: {max(values):.1f}%")
    print(f"{'─'*60}")
    return avg

In [52]:
import pynvml

In [53]:
def llm_response(llm, formatted_output, test_questions, stream: bool = True):
    response_output = []
    monitor = ResourceMonitor()

    try:
        pynvml.nvmlInit()
        gpu_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    except:
        gpu_handle = None

    print(f"\n{'='*100}")
    print(f"  RUNNING MODEL: {llm.model_name}")
    print(f"{'='*100}")

    for idx, output in enumerate(formatted_output, 1):
        query = output.get("query", "")
        text_context = output.get("text_context", "")
        images = output.get("images", [])

        if images:
            captions = [f"[Image {i+1} Caption] {img.get('caption','')}"
                       for i, img in enumerate(images) if img.get('caption')]
            if captions:
                text_context += "\n\n[Image Captions]\n" + "\n".join(captions)

        print(f"\n  QUERY #{idx}/{len(formatted_output)}")
        print(f"  {'─'*96}")
        print(f"  Context: {len(text_context):,} chars | Images: {len(images)}")

        start_total = time.perf_counter()

        # Get initial snapshot
        start_snap = monitor.get_snapshot()
        time.sleep(0.1)  # Small delay for CPU measurement

        gpu_start = pynvml.nvmlDeviceGetUtilizationRates(gpu_handle).gpu if gpu_handle else 0

        response_dict = llm.generate_response(
            query=query,
            context=text_context,
            images=images,
            stream=stream,  # Live streaming for clean output
            max_tokens=1000,
            temperature=0.4
        )

        # Get final measurements
        time.sleep(0.1)  # Small delay for CPU measurement
        end_snap = monitor.get_snapshot()

        gpu_end = pynvml.nvmlDeviceGetUtilizationRates(gpu_handle).gpu if gpu_handle else 0
        e2e_latency = round(time.perf_counter() - start_total, 4)

        # Calculate average CPU usage between start and end
        avg_cpu = (start_snap["cpu_percent"] + end_snap["cpu_percent"]) / 2
        avg_ram = (start_snap["ram_gb"] + end_snap["ram_gb"]) / 2
        avg_gpu = (gpu_start + gpu_end) / 2

        full_text = response_dict.get("response", "") if isinstance(response_dict, dict) else str(response_dict)

        print(f"\n  {'─'*96}")
        print(f"  METRICS:")
        print(f"       Inference Time: {response_dict.get('generation_time_sec', 0):.4f} s")
        print(f"       E2E Latency: {e2e_latency:.4f} s")
        print(f"       CPU Usage: {avg_cpu:.1f}%")
        print(f"       RAM Usage: {avg_ram:.3f} GB")
        print(f"       GPU Usage: {avg_gpu:.1f}%")
        print(f"  {'─'*96}")

        response_output.append({
            "id": test_questions[idx-1].get("id") if idx <= len(test_questions) else idx,
            "query": query,
            "context": text_context,
            "response": response_dict,
            "response_text": full_text,
            "images": images,
            "inference_time_sec": response_dict.get("generation_time_sec", 0),
            "e2e_latency_sec": e2e_latency,
            "avg_cpu_percent": round(avg_cpu, 2),
            "avg_ram_gb": round(avg_ram, 4),
            "avg_gpu_percent": round(avg_gpu, 2),
            "num_images": len(images)
        })

    print(f"\n{'='*100}")
    print(f"  COMPLETED MODEL: {llm.model_name}")
    print(f"{'='*100}")

    if gpu_handle:
        try:
            pynvml.nvmlShutdown()
        except:
            pass

    return response_output

In [54]:
import os
import re
import io
from datetime import datetime
from html import escape
from typing import List, Dict

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    PageBreak,
    Image as RLImage,
    KeepTogether,
    HRFlowable
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch

In [55]:
def export_retrieved_results_to_pdf(formatted_output, output_dir="../data/retrieval_results"):
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    filename = f"{output_dir}/retrieval_results_{timestamp}.pdf"

    doc = SimpleDocTemplate(filename, pagesize=A4, rightMargin=45, leftMargin=45,
                            topMargin=60, bottomMargin=50)
    elements = []
    styles = getSampleStyleSheet()

    title_style = ParagraphStyle("TitleStyle", parent=styles["Heading1"], fontSize=20, spaceAfter=20, textColor=colors.darkblue)
    section_style = ParagraphStyle("SectionStyle", parent=styles["Heading3"], fontSize=12, spaceAfter=6)
    body_style = ParagraphStyle("BodyStyle", parent=styles["Normal"], fontSize=10.5, leading=15, spaceAfter=8)
    caption_style = ParagraphStyle("CaptionStyle", parent=styles["Normal"], fontSize=9, textColor=colors.grey, leading=12, spaceAfter=10, alignment=1)

    elements.append(Paragraph("Retrieved Results Report", title_style))
    elements.append(Paragraph(f"Generated: {datetime.now().strftime('%d %B %Y, %I:%M %p')}", body_style))
    elements.append(Paragraph(f"Total Queries: {len(formatted_output)}", body_style))
    elements.append(Spacer(1, 0.4 * inch))
    elements.append(HRFlowable(width="100%", thickness=1, color=colors.grey))
    elements.append(Spacer(1, 0.5 * inch))
    elements.append(PageBreak())

    for idx, item in enumerate(formatted_output, 1):
        query = escape(item.get("query", ""))
        text_context = escape(item.get("text_context", "")).replace("\n", "<br/>")
        images = item.get("images", [])

        elements.append(Paragraph(f"Query {idx}", styles["Heading2"]))
        elements.append(Spacer(1, 0.2 * inch))
        elements.append(Paragraph("User Query", section_style))
        elements.append(Paragraph(query, body_style))
        elements.append(Paragraph("Retrieved Context", section_style))
        elements.append(Paragraph(text_context, body_style))

        if images:
            image_section = [Spacer(1, 0.3 * inch), Paragraph("Retrieved Image(s)", section_style), Spacer(1, 0.2 * inch)]
            for img_obj in images:
                pil_img = img_obj.get("image")
                caption = escape(img_obj.get("caption", ""))
                if pil_img is None:
                    continue
                img_buffer = io.BytesIO()
                pil_img.save(img_buffer, format="PNG")
                img_buffer.seek(0)
                rl_img = RLImage(img_buffer)
                scale = min(4.8 * inch / pil_img.size[0], 3.5 * inch / pil_img.size[1])
                rl_img.drawWidth = pil_img.size[0] * scale
                rl_img.drawHeight = pil_img.size[1] * scale
                rl_img.hAlign = "CENTER"
                image_section.append(rl_img)
                if caption:
                    image_section.append(Paragraph(f"<i>{caption}</i>", caption_style))
                image_section.append(Spacer(1, 0.35 * inch))
            elements.append(KeepTogether(image_section))

        elements.append(PageBreak())

    def add_page_number(canvas_obj, doc):
        canvas_obj.setFont("Helvetica", 9)
        canvas_obj.drawRightString(A4[0] - 40, 20, f"Page {doc.page}")

    doc.build(elements, onLaterPages=add_page_number)
    print(f"\n  ✓ Saved retrieval report: {filename}")

In [56]:
import os
import re
import io
from datetime import datetime
from xml.sax.saxutils import escape

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    PageBreak,
    Image as RLImage,
    KeepTogether,
    HRFlowable
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch
from reportlab.lib import colors

In [57]:
def export_results_to_pdf(results, model_name: str, metrics_summary: dict, output_dir="../data/results"):
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    safe_name = re.sub(r"[^\w\-]", "_", model_name)
    filename = f"{output_dir}/{safe_name}_rag_results_{timestamp}.pdf"

    doc = SimpleDocTemplate(filename, pagesize=A4, rightMargin=45, leftMargin=45,
                            topMargin=60, bottomMargin=50)
    elements = []
    styles = getSampleStyleSheet()

    title_style = ParagraphStyle("TitleStyle", parent=styles["Heading1"], fontSize=20, spaceAfter=20, textColor=colors.darkblue)
    section_style = ParagraphStyle("SectionStyle", parent=styles["Heading3"], fontSize=12, spaceAfter=6)
    body_style = ParagraphStyle("BodyStyle", parent=styles["Normal"], fontSize=10.5, leading=15, spaceAfter=8)
    metrics_style = ParagraphStyle("MetricsStyle", parent=styles["Normal"], fontSize=9.5, textColor=colors.darkgreen, spaceAfter=6)

    elements.append(Paragraph("RAG Evaluation Report - Traditional Vector RAG", title_style))
    elements.append(Paragraph(f"Model: {model_name}", body_style))
    elements.append(Paragraph(f"Generated: {datetime.now().strftime('%d %B %Y, %I:%M %p')}", body_style))
    elements.append(Spacer(1, 0.5 * inch))
    elements.append(HRFlowable(width="100%", thickness=1, color=colors.grey))
    elements.append(Spacer(1, 0.5 * inch))

    # Clean Grouped Metrics Summary
    elements.append(Paragraph("System Efficiency Metrics", styles["Heading2"]))
    elements.append(Spacer(1, 0.2 * inch))

    summary = f"""
    <b>Embedding & Indexing</b><br/>
    M1 Embedding Time : {metrics_summary.get('m1_embedding_time', 0):.4f} seconds<br/>
    M2 Index Size : {metrics_summary.get('m2_index_size', 0)} vectors<br/><br/>

    <b>Retrieval Quality</b><br/>
    M3 Retrieval Latency : {metrics_summary.get('m3_retrieval_latency', 0):.4f} seconds<br/>
    M4 Cosine Similarity : {metrics_summary.get('m4_cosine_similarity', 0):.4f}<br/>
    M5 Top-k Accuracy : {metrics_summary.get('m5_top_k_accuracy', 0):.2f} %<br/><br/>

    <b>ROUGE Scores</b><br/>
    M6 ROUGE-1 : {metrics_summary.get('m6_rouge1', 0):.4f}<br/>
    M7 ROUGE-2 : {metrics_summary.get('m7_rouge2', 0):.4f}<br/>
    M8 ROUGE-L : {metrics_summary.get('m8_rougeL', 0):.4f}<br/><br/>

    <b>Context & Generation</b><br/>
    M9 Context Length : {metrics_summary.get('m9_context_length', 0):.2f} characters<br/>
    M15 GT Coverage : {metrics_summary.get('m15_context_coverage', 0):.2f} %<br/><br/>

    <b>Performance</b><br/>
    M16 E2E Latency : {metrics_summary.get('m16_e2e_latency', 0):.4f} seconds<br/>
    M17 Throughput : {metrics_summary.get('m17_throughput', 0):.3f} queries/sec<br/>
    M18 CPU Usage : {metrics_summary.get('m18_cpu_usage', 0):.2f} %<br/>
    M19 RAM Usage : {metrics_summary.get('m19_ram_usage', 0):.3f} GB<br/>
    GPU Usage : {metrics_summary.get('gpu_usage', 0):.2f} %
    """
    elements.append(Paragraph(summary, metrics_style))
    elements.append(PageBreak())

    # Per-query details
    for idx, item in enumerate(results, 1):
        query = escape(item.get("query", ""))
        context = escape(item.get("context", "")).replace("\n", "<br/>")
        response_text = escape(item.get("response_text", "")).replace("\n", "<br/>")

        elements.append(Paragraph(f"Query {idx}", styles["Heading2"]))
        elements.append(Spacer(1, 0.2 * inch))
        elements.append(Paragraph("User Query", section_style))
        elements.append(Paragraph(query, body_style))
        elements.append(Paragraph("Retrieved Context", section_style))
        elements.append(Paragraph(context, body_style))
        elements.append(Paragraph("Model Answer", section_style))
        elements.append(Paragraph(response_text, body_style))

        metrics_text = f"""
        Inference Time : {item.get('inference_time_sec',0):.4f} sec<br/>
        E2E Latency : {item.get('e2e_latency_sec',0):.4f} sec<br/>
        CPU : {item.get('avg_cpu_percent',0):.1f} %<br/>
        RAM : {item.get('avg_ram_gb',0):.3f} GB<br/>
        GPU : {item.get('avg_gpu_percent',0):.1f} %<br/>
        Images Used : {item.get('num_images',0)}
        """
        elements.append(Paragraph(metrics_text, metrics_style))

        if item.get("images"):
            image_section = [Spacer(1, 0.3 * inch), Paragraph("Retrieved Image(s)", section_style), Spacer(1, 0.2 * inch)]
            for img_obj in item["images"]:
                pil_img = img_obj.get("image")
                caption = escape(img_obj.get("caption", ""))
                if pil_img is None:
                    continue
                img_buffer = io.BytesIO()
                pil_img.save(img_buffer, format="PNG")
                img_buffer.seek(0)
                rl_img = RLImage(img_buffer)
                scale = min(4.8 * inch / pil_img.size[0], 3.5 * inch / pil_img.size[1])
                rl_img.drawWidth = pil_img.size[0] * scale
                rl_img.drawHeight = pil_img.size[1] * scale
                rl_img.hAlign = "CENTER"
                image_section.append(rl_img)
                if caption:
                    image_section.append(Paragraph(f"<i>{caption}</i>", ParagraphStyle("CaptionStyle", parent=styles["Normal"], fontSize=9, textColor=colors.grey, alignment=1)))
                image_section.append(Spacer(1, 0.35 * inch))
            elements.append(KeepTogether(image_section))
        elements.append(PageBreak())

    def add_page_number(canvas_obj, doc):
        canvas_obj.setFont("Helvetica", 9)
        canvas_obj.drawRightString(A4[0] - 40, 20, f"Page {doc.page}")

    doc.build(elements, onLaterPages=add_page_number)
    print(f"\n  ✓ Saved evaluation report: {filename}")

In [58]:
def main(test_questions):
    """
    Main RAG pipeline execution.
    Flow: Load → Chunk → Embed → Index → Retrieve → Generate → Evaluate → Export
    """

    # =============================================================================
    # PHASE 1: DOCUMENT PROCESSING
    # =============================================================================
    print("\n" + "="*100)
    print("PHASE 1: LOADING DOCUMENTS & CHUNKING")
    print("="*100)

    # Load PDFs and extract text blocks + images
    pages = loading_pdf(dir_path="../data/pdf")

    # Chunk text using bbox-based chunking with token limits
    chunks = bbox_chunker(pages, max_tokens=384, overlap_tokens=96)

    # Build image objects with captions
    image_objects = build_image_objects(pages)

    # =============================================================================
    # PHASE 2: EMBEDDING & INDEXING
    # =============================================================================
    print("\n" + "="*100)
    print("PHASE 2: EMBEDDING & VECTOR STORE INITIALIZATION")
    print("="*100)

    # Initialize embedding models
    text_embedder = TextEmbeddingModel()
    image_embedder = ImageEmbeddingModel()

    # Create embeddings (returns embeddings + timing + stats)
    text_embeddings, text_time, text_stats = text_embedder.embed_documents(chunks)
    image_embeddings, image_time, image_stats = image_embedder.embed_image(image_objects)

    # Initialize vector stores with silent=True to suppress printing
    text_db = VectorStore(collection_name="vector_text", silent=True)
    image_db = VectorStore(collection_name="vector_image", silent=True)

    # Add documents to stores
    text_db.add_documents(chunks, text_embeddings)
    image_db.add_documents(image_objects, image_embeddings)

    # =============================================================================
    # PHASE 3: RETRIEVAL SETUP
    # =============================================================================

    # Initialize retriever with hybrid search (text + image)
    retriever = RetrievalRag(
        image_embedder=image_embedder,
        text_embedder=text_embedder,
        image_vectordb=image_db,
        text_vectordb=text_db
    )

    # Initialize context formatter
    formatter = ContextFormatter(
        max_text_chunks=3,
        max_images=1
    )

    # =============================================================================
    # PHASE 4: MODEL EVALUATION
    # =============================================================================

    models = ["gemma3:4b"]

    for model_name in models:
        print("\n" + "="*100)
        print(f"PHASE 4: EVALUATING MODEL - {model_name}")
        print("="*100)

        # Initialize LLM
        llm = LocalLLM(
            model_name=model_name,
            text_embedder=text_embedder
        )

        # Containers for per-query results
        formatted_output = []
        retrieval_times = []
        cosine_sims = []
        retrieval_for_m5 = []

        # -------------------------------------------------------------------------
        # Step 4.1: Retrieve context for all questions
        # -------------------------------------------------------------------------
        print(f"\n  Retrieving context for {len(test_questions)} questions...")

        for q in test_questions:
            # Time the retrieval
            start = time.perf_counter()
            out = retriever.retrieve(q["question"])
            retrieval_times.append(time.perf_counter() - start)

            # Store cosine similarity from retrieval
            cosine_sims.append(
                out["retrieval_metrics"].get("cosine_similarity", 0.0)
            )

            # Format for LLM consumption
            formatted = formatter.format(out)
            formatted["query"] = q["question"]
            formatted_output.append(formatted)

            # Store for M5 calculation
            retrieval_for_m5.append({
                "id": q.get("id"),
                "result": out
            })

        # -------------------------------------------------------------------------
        # Step 4.2: Generate LLM responses
        # -------------------------------------------------------------------------
        per_query_results = llm_response(
            llm=llm,
            formatted_output=formatted_output,
            test_questions=test_questions,
            stream=True
        )

        # =============================================================================
        # PHASE 5: METRICS CALCULATION
        # =============================================================================
        print("\n" + "="*100)
        print("PHASE 5: CALCULATING METRICS")
        print("="*100)

        metrics_summary = {}

        # Efficiency Metrics (M1, M2) - call functions here
        metrics_summary["m1_embedding_time"] = compute_embedding_time(text_time, image_time)
        metrics_summary["m2_index_size"] = compute_index_size(text_db, image_db)

        # Retrieval Quality Metrics (M3-M5)
        metrics_summary["m3_retrieval_latency"] = compute_retrieval_latency(retrieval_times)
        metrics_summary["m4_cosine_similarity"] = compute_cosine_similarity(cosine_sims)
        metrics_summary["m5_top_k_accuracy"] = compute_top_k_accuracy(
            retrieval_for_m5,
            test_questions
        )

        # Generation Quality Metrics (M6-M8, M15)
        metrics_summary["m6_rouge1"] = compute_rouge1(per_query_results)
        metrics_summary["m7_rouge2"] = compute_rouge2(per_query_results)
        metrics_summary["m8_rougeL"] = compute_rougeL(per_query_results)
        metrics_summary["m15_context_coverage"] = compute_context_coverage(per_query_results)

        # Context & System Metrics (M9, M16-M19, GPU)
        metrics_summary["m9_context_length"] = compute_context_length(formatted_output)
        metrics_summary["m16_e2e_latency"] = compute_e2e_latency(per_query_results)
        metrics_summary["m17_throughput"] = compute_throughput(per_query_results)
        metrics_summary["m18_cpu_usage"] = compute_cpu_usage(per_query_results)
        metrics_summary["m19_ram_usage"] = compute_ram_usage(per_query_results)
        metrics_summary["gpu_usage"] = compute_gpu_usage(per_query_results)

        # =============================================================================
        # PHASE 6: RESULTS DISPLAY
        # =============================================================================
        print("\n" + "="*100)
        print("PHASE 6: FINAL METRICS SUMMARY")
        print("="*100)

        # Embedding Details Table (using captured stats from embedding functions)
        print(f"\n  {'─'*78}")
        print(f"  {'EMBEDDING STATISTICS':^76}")
        print(f"  {'─'*78}")
        print(f"  {'Text Embeddings (BGE-large-en-v1.5)':<50} {'Dim: ' + str(text_stats['dimension']):>24}")
        print(f"    Documents processed:  {text_stats['count']}")
        print(f"    Total time:           {text_stats['total_time']:.4f} seconds")
        print(f"    Avg time/document:    {text_stats['avg_time']:.4f} seconds")
        print(f"")
        print(f"  {'Image Embeddings (ViT-L-14)':<50} {'Dim: ' + str(image_stats['dimension']):>24}")
        print(f"    Images processed:     {image_stats['count']}")
        print(f"    Total time:           {image_stats['total_time']:.4f} seconds")
        print(f"    Avg time/image:       {image_stats['avg_time']:.4f} seconds")
        print(f"  {'─'*78}")
        print(f"  {'TOTAL EMBEDDING TIME (M1)':<50} {metrics_summary['m1_embedding_time']:.4f} seconds")
        print(f"  {'─'*78}")

        # Main Metrics Table
        print(f"\n  ┌{'─'*58}┐")
        print(f"  │{'METRIC':<30}│{'VALUE':>26}│")
        print(f"  ├{'─'*58}┤")

        # Efficiency
        print(f"  │{'─'*56}│")
        print(f"  │{'EFFICIENCY':^56}│")
        print(f"  │{'─'*56}│")
        print(f"  │{'M1 Embedding Time':<30}│{metrics_summary['m1_embedding_time']:>24.4f} s│")
        print(f"  │{'M2 Index Size':<30}│{metrics_summary['m2_index_size']:>26}│")

        # Retrieval Quality
        print(f"  │{'─'*56}│")
        print(f"  │{'RETRIEVAL QUALITY':^56}│")
        print(f"  │{'─'*56}│")
        print(f"  │{'M3 Retrieval Latency':<30}│{metrics_summary['m3_retrieval_latency']:>24.4f} s│")
        print(f"  │{'M4 Cosine Similarity':<30}│{metrics_summary['m4_cosine_similarity']:>26.4f}│")
        print(f"  │{'M5 Top-k Accuracy':<30}│{metrics_summary['m5_top_k_accuracy']:>25.2f} %│")

        # Generation Quality
        print(f"  │{'─'*56}│")
        print(f"  │{'GENERATION QUALITY':^56}│")
        print(f"  │{'─'*56}│")
        print(f"  │{'M6 ROUGE-1':<30}│{metrics_summary['m6_rouge1']:>26.4f}│")
        print(f"  │{'M7 ROUGE-2':<30}│{metrics_summary['m7_rouge2']:>26.4f}│")
        print(f"  │{'M8 ROUGE-L':<30}│{metrics_summary['m8_rougeL']:>26.4f}│")
        print(f"  │{'M15 Context Coverage':<30}│{metrics_summary['m15_context_coverage']:>25.2f} %│")

        # Context & Performance
        print(f"  │{'─'*56}│")
        print(f"  │{'CONTEXT & PERFORMANCE':^56}│")
        print(f"  │{'─'*56}│")
        print(f"  │{'M9 Context Length':<30}│{metrics_summary['m9_context_length']:>23.2f} ch│")
        print(f"  │{'M16 E2E Latency':<30}│{metrics_summary['m16_e2e_latency']:>24.4f} s│")
        print(f"  │{'M17 Throughput':<30}│{metrics_summary['m17_throughput']:>25.3f} q/s│")
        print(f"  │{'M18 CPU Usage':<30}│{metrics_summary['m18_cpu_usage']:>25.2f} %│")
        print(f"  │{'M19 RAM Usage':<30}│{metrics_summary['m19_ram_usage']:>24.3f} GB│")
        print(f"  │{'GPU Usage':<30}│{metrics_summary['gpu_usage']:>25.2f} %│")

        print(f"  └{'─'*58}┘")

        # =============================================================================
        # PHASE 7: EXPORT RESULTS
        # =============================================================================
        print("\n" + "="*100)
        print("PHASE 7: EXPORTING RESULTS")
        print("="*100)

        export_results_to_pdf(
            results=per_query_results,
            model_name=model_name,
            metrics_summary=metrics_summary
        )

    # =============================================================================
    # COMPLETION
    # =============================================================================
    print("\n" + "="*100)
    print("PIPELINE COMPLETED SUCCESSFULLY")
    print("="*100)

In [59]:
if __name__ == "__main__":
    main(TEST_QUESTIONS)


PHASE 1: LOADING DOCUMENTS & CHUNKING

  PDF LOADING
  Directory: ..\data\pdf
  Found 3 PDF file(s)

  Loading PDFs...
  ----------------------------------------------------------------------------
  [1/3] Loading: highlights_of_hubbles_exploration_of_the_universe.pdf
       Size: 9.85 MB
       ✓ Extracted 0 text blocks, 1 images
  [2/3] Loading: mars-science-laboratory.pdf
       Size: 1.44 MB
       ✓ Extracted 72 text blocks, 0 images
  [3/3] Loading: Voyager Grand Tour PDF.pdf
       Size: 0.67 MB
       ✓ Extracted 2 text blocks, 0 images
  ----------------------------------------------------------------------------

  SUMMARY:
       Total size: 11.96 MB
       Total pages extracted: 30
       Successful: 3/3


  Created 71 chunks from 30 pages (max_tokens=384, overlap_tokens=96)

PHASE 2: EMBEDDING & VECTOR STORE INITIALIZATION


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BGE running on NVIDIA GeForce RTX 3050 6GB Laptop GPU.
Embedding dimension of BAAI/bge-large-en-v1.5 is 1024
OpenCLIP running on NVIDIA GeForce RTX 3050 6GB Laptop GPU.
Embedding dimension of ViT-L-14 is 768


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Reranker initialized

PHASE 4: EVALUATING MODEL - gemma3:4b
Ollama server started

  Retrieving context for 5 questions...

  RUNNING MODEL: gemma3:4b

  QUERY #1/5
  ────────────────────────────────────────────────────────────────────────────────────────────────
  Context: 3,771 chars | Images: 1

QUERY: Why does the visual geometry prove that gravity assists fundamentally redirect trajectories rather than merely accelerate the spacecraft, and what irreversible decision did this force upon Voyager 1 at Saturn?
ANSWER:
--------------------------------------------------------------------------------
Okay, let’s analyze the provided text and images to answer the question about gravity assists and Voyager 1’s decision at Saturn.

1) **Textual Evidence Assessment:**

The text describes the Grand Tour mission, utilizing gravity assists to navigate between planets. It states: "A very rare planetary alignment would occur in the late 1970’s allowing a spacecraft to visit all the outer planets 